In [ ]:
# pip install pydrive2 gdown google-api-python-client

import subprocess
import sys
import yaml
from pathlib import Path
from typing import Dict, Any, Optional

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

REPO_ROOT = Path(__file__).resolve().parents[1]
CONFIG_PATH = REPO_ROOT / "configs" / "data_registry.yaml"
DRIVE_CONFIG_PATH = REPO_ROOT / "configs" / "drive_config.yaml"
CLIENT_SECRETS_PATH = REPO_ROOT / "configs" / "client_secrets.json"
DRIVE_CREDENTIALS_PATH = REPO_ROOT / "configs" / "pydrive_credentials.json"

_DATA_REGISTRY: Dict[str, Any] = {}
_LOADED = False

# Cached Google Drive client + root folder id
_DRIVE: Optional[GoogleDrive] = None
_DRIVE_ROOT_ID: Optional[str] = None


def resync_registry() -> None:
    """
    Automatically (re)build data_registry.yaml by calling data_registry_sync.py.

    Safe to call at the start of any script that needs registry-driven IO.
    Relies on our shared Google auth (which caches credentials).
    """
    sync_script = REPO_ROOT / "common_scripts" / "data_registry_sync.py"
    if not sync_script.exists():
        print(
            f"[data_io] WARNING: {sync_script} not found; "
            f"cannot resync data registry."
        )
        return

    print("[data_io] Resyncing data registry from Google Drive...")
    cmd = [sys.executable, str(sync_script), "--mode", "resync"]
    subprocess.run(cmd, check=True)
    global _LOADED
    _LOADED = False  # force reload on next _load_registry
    print("[data_io] Registry resync complete.")


def _load_registry(force: bool = False) -> None:
    """Lazy-load data_registry.yaml into memory."""
    global _DATA_REGISTRY, _LOADED
    if _LOADED and not force:
        return
    if not CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"data_registry.yaml not found at {CONFIG_PATH}. "
            "Run common_scripts/data_registry_sync.py or call resync_registry() first."
        )
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        _DATA_REGISTRY = yaml.safe_load(f) or {}
    _LOADED = True


def get_entry(namespace: str, key: str) -> dict:
    _load_registry()
    try:
        ns = _DATA_REGISTRY[namespace]
    except KeyError:
        raise KeyError(f"No namespace '{namespace}' in data registry.")
    try:
        return ns[key]
    except KeyError:
        raise KeyError(f"No registry entry for {namespace}.{key}")


def ensure_local(namespace: str, key: str) -> Path:
    """
    Ensure the data file for (namespace, key) exists locally.
    If not, and drive_file_id is present, download it via gdown.
    """
    entry = get_entry(namespace, key)
    local_rel = entry["local_path"]
    local_path = REPO_ROOT / local_rel
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        return local_path

    file_id = entry.get("drive_file_id")
    if not file_id:
        raise FileNotFoundError(
            f"{local_path} is missing and no drive_file_id specified in registry "
            f"for {namespace}.{key}"
        )

    print(f"[data_io] Downloading {namespace}.{key} from Drive -> {local_path}")
    cmd = [
        "gdown",
        "--id",
        file_id,
        "-O",
        str(local_path),
    ]
    subprocess.run(cmd, check=True)
    return local_path


def ensure_local_path(rel_path: str) -> Path:
    """
    Convenience: ensure a file at `rel_path` exists locally, using registry to
    find its drive_file_id (if any).
    """
    _load_registry()
    rel_path = str(Path(rel_path))
    for ns_name, ns_dict in _DATA_REGISTRY.items():
        if not isinstance(ns_dict, dict):
            continue
        for key, entry in ns_dict.items():
            if not isinstance(entry, dict):
                continue
            if entry.get("local_path") == rel_path:
                return ensure_local(ns_name, key)
    raise KeyError(
        f"No registry entry found with local_path={rel_path} in {CONFIG_PATH}"
    )


def _load_drive_root_id() -> str:
    """
    Read the shared Drive root folder ID from drive_config.yaml.
    This is the folder that corresponds to repo_root/data on Drive.
    """
    if not DRIVE_CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Missing drive_config.yaml at {DRIVE_CONFIG_PATH}. "
            "Create it with a `drive_root_folder_id` field."
        )
    with open(DRIVE_CONFIG_PATH, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    root_id = cfg.get("drive_root_folder_id")
    if not root_id:
        raise ValueError("drive_root_folder_id missing in drive_config.yaml")
    return root_id


def _get_drive_and_root():
    """
    Return a cached (GoogleDrive client, drive_root_folder_id) pair.

    This:
      - Loads client_secrets.json for OAuth config
      - Loads cached pydrive_credentials.json if present
      - Only calls LocalWebserverAuth() (browser) if credentials are missing
        or expired, and then saves them for future use.
    """
    global _DRIVE, _DRIVE_ROOT_ID
    if _DRIVE is not None and _DRIVE_ROOT_ID is not None:
        return _DRIVE, _DRIVE_ROOT_ID

    gauth = GoogleAuth()
    # Configure OAuth client
    gauth.LoadClientConfigFile(str(CLIENT_SECRETS_PATH))

    # Try to load cached credentials (no browser)
    if DRIVE_CREDENTIALS_PATH.exists():
        gauth.LoadCredentialsFile(str(DRIVE_CREDENTIALS_PATH))

    if gauth.credentials is None or gauth.access_token_expired:
        # Only in this case do we actually open the browser
        print("[data_io] Performing Google OAuth (one-time)...")
        gauth.LocalWebserverAuth()
        # Cache credentials for later runs
        gauth.SaveCredentialsFile(str(DRIVE_CREDENTIALS_PATH))
    else:
        # Use existing valid creds silently
        print("[data_io] Using cached Google Drive credentials.")

    drive = GoogleDrive(gauth)
    root_id = _load_drive_root_id()

    _DRIVE = drive
    _DRIVE_ROOT_ID = root_id
    return _DRIVE, _DRIVE_ROOT_ID


def upload_to_drive(local_path: Path) -> str:
    """
    Upload or update `local_path` under the shared Drive 'data/' tree,
    mirroring the repo's data/ subfolders.

    Returns the file_id of the uploaded Google Drive file.
    """
    local_path = Path(local_path)
    if not local_path.exists():
        raise FileNotFoundError(f"Cannot upload missing file: {local_path}")

    rel = local_path.relative_to(REPO_ROOT)
    if not rel.parts or rel.parts[0] != "data":
        raise ValueError(
            f"upload_to_drive currently only supports files under 'data/' "
            f"(got: {rel})"
        )

    rel_under_data = rel.parts[1:]
    if not rel_under_data:
        raise ValueError(
            f"Unexpected path with no components under data/: {rel}"
        )

    drive, root_id = _get_drive_and_root()
    parent_id = root_id

    def _q_escape(s: str) -> str:
        return s.replace("'", "\\'")

    # Create/find folder chain under Drive root
    for folder_name in rel_under_data[:-1]:
        folder_name_q = _q_escape(folder_name)
        query = (
            f"'{parent_id}' in parents and "
            f"title = '{folder_name_q}' and "
            "mimeType = 'application/vnd.google-apps.folder' and trashed=false"
        )
        folder_list = drive.ListFile({"q": query}).GetList()
        if folder_list:
            parent_id = folder_list[0]["id"]
        else:
            folder_metadata = {
                "title": folder_name,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [{"id": parent_id}],
            }
            folder = drive.CreateFile(folder_metadata)
            folder.Upload()
            parent_id = folder["id"]

    filename = rel_under_data[-1]
    filename_q = _q_escape(filename)
    query = (
        f"'{parent_id}' in parents and "
        f"title = '{filename_q}' and trashed=false"
    )
    existing = drive.ListFile({"q": query}).GetList()

    if existing:
        gfile = existing[0]
        print(f"[data_io] Updating existing Drive file for {rel}")
    else:
        gfile = drive.CreateFile(
            {"title": filename, "parents": [{"id": parent_id}]}
        )
        print(f"[data_io] Creating new Drive file for {rel}")

    gfile.SetContentFile(str(local_path))
    gfile.Upload()
    file_id = gfile["id"]
    print(f"[data_io] Uploaded {rel} to Drive (file_id={file_id})")
    return file_id

def remote_file_exists_by_rel_path(rel_path: str) -> bool:
    """
    Check if a file with the given repo-relative path exists on Drive
    under the shared 'data/' tree.

    rel_path is something like: 'data/processed/Category/file.parquet'
    or 'data/locks/03_user_features/All_Beauty.lock'
    """
    from pathlib import Path

    drive, root_id = _get_drive_and_root()
    rel = Path(rel_path)

    # Ensure path is anchored under 'data/...'
    parts = rel.parts
    if parts and parts[0] == "data":
        parts = parts[1:]
    else:
        parts = ("data",) + parts

    if not parts:
        return False

    folder_parts, filename = parts[:-1], parts[-1]
    parent_id = root_id

    def _q_escape(s: str) -> str:
        return s.replace("'", "\\'")

    # Walk down folder chain
    for folder_name in folder_parts:
        folder_name_q = _q_escape(folder_name)
        query = (
            f"'{parent_id}' in parents and "
            f"title = '{folder_name_q}' and "
            "mimeType = 'application/vnd.google-apps.folder' and trashed=false"
        )
        flist = drive.ListFile({"q": query}).GetList()
        if not flist:
            return False
        parent_id = flist[0]["id"]

    filename_q = _q_escape(filename)
    query = (
        f"'{parent_id}' in parents and "
        f"title = '{filename_q}' and trashed=false"
    )
    files = drive.ListFile({"q": query}).GetList()
    return bool(files)


def delete_remote_by_rel_path(rel_path: str) -> None:
    """
    Delete (if present) a remote file at the given repo-relative path from Drive.

    rel_path example: 'data/locks/03_user_features/All_Beauty.lock'
    """
    from pathlib import Path

    drive, root_id = _get_drive_and_root()
    rel = Path(rel_path)

    parts = rel.parts
    if parts and parts[0] == "data":
        parts = parts[1:]
    else:
        parts = ("data",) + parts

    if not parts:
        return

    folder_parts, filename = parts[:-1], parts[-1]
    parent_id = root_id

    def _q_escape(s: str) -> str:
        return s.replace("'", "\\'")

    # Walk folders
    for folder_name in folder_parts:
        folder_name_q = _q_escape(folder_name)
        query = (
            f"'{parent_id}' in parents and "
            f"title = '{folder_name_q}' and "
            "mimeType = 'application/vnd.google-apps.folder' and trashed=false"
        )
        flist = drive.ListFile({"q": query}).GetList()
        if not flist:
            return  # folder chain missing -> nothing to delete
        parent_id = flist[0]["id"]

    filename_q = _q_escape(filename)
    query = (
        f"'{parent_id}' in parents and "
        f"title = '{filename_q}' and trashed=false"
    )
    files = drive.ListFile({"q": query}).GetList()
    for f in files:
        f.Delete()

In [ ]:
#!/usr/bin/env python
"""
Scan local data/ and shared Google Drive folder and auto-generate configs/data_registry.yaml.

Usage:

  python common_scripts/data_registry_sync.py --mode resync

This will:
  - connect to Google Drive
  - walk the shared folder tree (assumed to mirror `data/` structure)
  - scan local `data/` tree
  - build / overwrite configs/data_registry.yaml so that `data_io.ensure_local(...)`
    knows how to download files by key.
"""

import argparse
from pathlib import Path
from typing import Dict, Tuple
import sys
import yaml
from pydrive2.drive import GoogleDrive

# Make sure we can import siblings (data_io.py) no matter where we run it from
THIS_DIR = Path(__file__).resolve().parent
if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

from data_io import _get_drive_and_root

REPO_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = REPO_ROOT / "data"
REGISTRY_PATH = REPO_ROOT / "configs" / "data_registry.yaml"
DRIVE_CFG_PATH = REPO_ROOT / "configs" / "drive_config.yaml"
CLIENT_SECRETS_PATH = REPO_ROOT / "configs" / "client_secrets.json"

def load_drive_root_id() -> str:
    if not DRIVE_CFG_PATH.exists():
        raise FileNotFoundError(
            f"Missing drive_config.yaml at {DRIVE_CFG_PATH}. "
            "Create it with a `drive_root_folder_id` field."
        )
    with open(DRIVE_CFG_PATH, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    root_id = cfg.get("drive_root_folder_id")
    if not root_id:
        raise ValueError("drive_root_folder_id missing in drive_config.yaml")
    return root_id


def get_drive() -> GoogleDrive:
    """
    Use the shared Drive client from data_io, which caches credentials and only
    does browser-based auth when necessary.
    """
    drive, _ = _get_drive_and_root()
    return drive


def walk_drive_tree(drive: GoogleDrive, root_folder_id: str) -> Dict[str, str]:
    """
    Recursively walk the shared Drive folder and return mapping:
      'data/processed/Category/file.ext' -> file_id
    We assume the Drive folder mirrors the `data/` substructure,
    so root_folder_id corresponds to `data/`.
    """
    relpath_to_id: Dict[str, str] = {}

    def _walk(folder_id: str, curr_rel: Path):
        # List children of this folder
        file_list = drive.ListFile(
            {"q": f"'{folder_id}' in parents and trashed=false"}
        ).GetList()

        for f in file_list:
            name = f["title"]
            mime = f["mimeType"]
            fid = f["id"]
            if mime == "application/vnd.google-apps.folder":
                _walk(fid, curr_rel / name)
            else:
                rel_path = Path("data") / curr_rel / name  # data/... path
                relpath_to_id[str(rel_path)] = fid

    _walk(root_folder_id, Path())
    return relpath_to_id


def scan_local_data() -> Dict[str, bool]:
    """
    Return set-like dict:
      'data/processed/.../file.ext' -> True
    for all local files under data/
    """
    local_map: Dict[str, bool] = {}
    if not DATA_DIR.exists():
        return local_map

    for p in DATA_DIR.rglob("*"):
        if p.is_file():
            rel = p.relative_to(REPO_ROOT)
            local_map[str(rel)] = True
    return local_map


def infer_namespace_key(rel_path: str) -> Tuple[str, str]:
    """
    Heuristic to map a relative path like:
      'data/processed/Automotive/user_counts_Automotive.parquet'
    to (namespace='processed', key='user_counts_Automotive').

    This matches how data_io currently expects the registry to be structured.
    """
    p = Path(rel_path)
    parts = p.parts  # ('data','processed','Automotive','user_counts_Automotive.parquet',...)

    if len(parts) >= 3 and parts[1] == "processed":
        namespace = "processed"
        key = p.stem  # e.g. 'user_counts_Automotive', 'top_users'
        return namespace, key

    if len(parts) >= 3 and parts[1] == "raw":
        namespace = "raw"
        key = p.name
        return namespace, key

    # global files under data/global/
    if len(parts) >= 3 and parts[1] == "global":
        namespace = "processed"
        key = p.stem
        return namespace, key

    # lock files under data/locks/...
    if len(parts) >= 3 and parts[1] == "locks":
        namespace = "locks"
        key = p.stem
        return namespace, key

    return "", ""


def build_registry(
    drive_root_id: str,
    local_map: Dict[str, bool],
    remote_map: Dict[str, str],
) -> Dict:
    """
    Build the full registry dict to dump as YAML.
    """
    registry = {
        "drive_root_folder_id": drive_root_id,
        "raw": {},
        "processed": {},
        "locks": {},
    }

    all_paths = sorted(set(local_map.keys()) | set(remote_map.keys()))

    for rel in all_paths:
        ns, key = infer_namespace_key(rel)
        if not ns or not key:
            # Unknown / unhandled location -> skip
            continue

        ns_dict = registry.setdefault(ns, {})
        entry = ns_dict.setdefault(key, {})
        entry["local_path"] = rel
        fid = remote_map.get(rel)
        if fid:
            entry["drive_file_id"] = fid

    return registry


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Sync local data/ and shared Google Drive into data_registry.yaml"
    )
    parser.add_argument(
        "--mode",
        choices=["resync"],
        default="resync",
        help="Currently only 'resync' is supported: rebuild the registry from scratch.",
    )
    args = parser.parse_args()

    drive_root_id = load_drive_root_id()
    drive = get_drive()

    print(">>> Scanning Google Drive shared data folder...")
    remote_map = walk_drive_tree(drive, drive_root_id)
    print(f"    Found {len(remote_map)} remote files under shared drive root.")

    print(">>> Scanning local data/ tree...")
    local_map = scan_local_data()
    print(f"    Found {len(local_map)} local files under data/")

    print(">>> Building data registry...")
    registry = build_registry(drive_root_id, local_map, remote_map)

    REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(REGISTRY_PATH, "w", encoding="utf-8") as f:
        yaml.safe_dump(registry, f, sort_keys=True)

    print(f"[OK] Wrote registry to {REGISTRY_PATH}")
    print("You can now use data_io.ensure_local(...) from your scripts.")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
"""
Preprocess Amazon-Reviews-2023 categories.

For each category:
  - Download / fetch review + meta .jsonl.gz files (Drive or UCSD)
  - Stream and parse the review file
  - Compute per-user purchase counts
  - Compute basic EDA stats (ratings, helpful votes, user purchase counts)
  - Stream meta file for simple item-level stats
  - Saves:
      data/processed/<Category>/user_counts_<Category>.parquet
      data/processed/<Category>/review_stats_<Category>.json
      data/processed/<Category>/meta_stats_<Category>.json
      data/processed/<Category>/*_hist_<Category>.png
  - By default, deletes local raw gz files when done (cleanup_raw=True).
"""

# pip install requests pandas matplotlib pyarrow

import argparse
import gzip
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import pandas as pd
import requests

# Make sure we can import siblings (data_io.py)
THIS_DIR = Path(__file__).resolve().parent
if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

from data_io import (  # noqa: E402
    ensure_local,
    ensure_local_path,
    resync_registry,
    upload_to_drive,
)


# Config / paths

REVIEW_URL_TEMPLATE = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/"
    "review_categories/{category}.jsonl.gz"
)
META_URL_TEMPLATE = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/"
    "meta_categories/meta_{category}.jsonl.gz"
)


def get_repo_root() -> Path:
    """Return repository root (one level above common_scripts)."""
    return Path(__file__).resolve().parents[1]


# Download helpers

def download_if_needed(url: str, dest: Path, force: bool = False) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not force:
        print(f"  [exists] {dest}")
        return

    print(f"  [download{' (force)' if force else ''}] {url} -> {dest}")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
    print("  [ok] download complete")


def ensure_raw_gzip_or_download(
    path: Path, url: str, allow_download: bool, repo_root: Path
) -> None:
    """
    Ensure a raw gzip exists locally by:
      1) Checking local path
      2) Trying Drive via ensure_local_path (using relative path)
      3) Falling back to direct HTTP download from UCSD if allowed
    """
    if path.exists():
        return

    rel = str(path.relative_to(repo_root))
    try:
        print(f"  [data_io] trying Drive for {rel}")
        ensure_local_path(rel)
        if path.exists():
            return
    except Exception:
        # No registry entry or download failed -> fall back
        pass

    if allow_download:
        download_if_needed(url, path, force=False)
    else:
        raise FileNotFoundError(
            f"Missing {path} and --no-download is set; "
            "no Drive entry or HTTP download attempted."
        )


def ensure_outputs_from_drive(paths: List[Path], repo_root: Path) -> None:
    """
    For each expected output, if it's missing locally but exists in Drive
    (according to registry), pull it down so we can skip work.
    """
    for p in paths:
        if p.exists():
            continue
        rel = str(p.relative_to(repo_root))
        try:
            ensure_local_path(rel)
        except Exception:
            # No registry entry or download failed – ignore.
            pass


# ---------------------------------------------------------------------------
# EDA & processing
# ---------------------------------------------------------------------------

def process_review_file(
    gz_path: Path,
    category: str,
    max_helpful_bucket: int = 10,
) -> Dict:
    print(f"  [reviews] parsing {gz_path}")

    user_counts = defaultdict(int)
    rating_hist = Counter()
    helpful_hist = Counter()
    n_reviews = 0
    n_verified = 0

    with gzip.open(gz_path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            user_id = obj.get("user_id")
            if user_id is None:
                continue

            user_counts[user_id] += 1
            n_reviews += 1

            rating = obj.get("rating")
            try:
                if rating is not None:
                    rating_hist[float(rating)] += 1
            except (TypeError, ValueError):
                pass

            # Helpful votes
            hv = obj.get("helpful_votes", 0)
            try:
                hv = int(hv)
            except (TypeError, ValueError):
                hv = 0
            hv_clipped = min(hv, max_helpful_bucket)
            helpful_hist[hv_clipped] += 1

            # Verified purchase
            if obj.get("verified_purchase"):
                n_verified += 1

    print(f"  [reviews] n_reviews={n_reviews:,}, n_users={len(user_counts):,}")
    return {
        "user_counts": user_counts,
        "rating_hist": rating_hist,
        "helpful_hist": helpful_hist,
        "n_reviews": n_reviews,
        "n_verified": n_verified,
    }


def process_meta_file(gz_path: Path) -> Dict:
    print(f"  [meta] parsing {gz_path}")

    n_items = 0
    rating_number_hist = Counter()
    price_count = 0
    price_sum = 0.0
    price_sum_sq = 0.0

    with gzip.open(gz_path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            n_items += 1

            # rating_number
            rn = obj.get("rating_number")
            try:
                if rn is not None:
                    rn = int(rn)
                    rating_number_hist[rn] += 1
            except (TypeError, ValueError):
                pass

            # price
            price = obj.get("price")
            try:
                if price is not None and price != "None":
                    p = float(price)
                    price_count += 1
                    price_sum += p
                    price_sum_sq += p * p
            except (TypeError, ValueError):
                pass

    print(f"  [meta] n_items={n_items:,}")
    return {
        "n_items": n_items,
        "rating_number_hist": rating_number_hist,
        "price_count": price_count,
        "price_sum": price_sum,
        "price_sum_sq": price_sum_sq,
    }


# Plot helpers

def save_rating_hist_plot(rating_hist: Counter, out_path: Path, title: str) -> None:
    if not rating_hist:
        print("  [plot] no ratings to plot")
        return
    items = sorted(rating_hist.items())
    xs = [k for k, _ in items]
    ys = [v for _, v in items]

    plt.figure()
    plt.bar(xs, ys)
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()
    print(f"  [plot] saved {out_path}")


def save_helpful_hist_plot(helpful_hist: Counter, out_path: Path, title: str) -> None:
    if not helpful_hist:
        print("  [plot] no helpful_votes to plot")
        return
    items = sorted(helpful_hist.items())
    xs = [k for k, _ in items]
    ys = [v for _, v in items]

    plt.figure()
    plt.bar(xs, ys)
    plt.xlabel("Helpful votes (clipped)")
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()
    print(f"  [plot] saved {out_path}")


def save_user_purchases_hist_plot(user_counts: Dict[str, int], out_path: Path, title: str) -> None:
    if not user_counts:
        print("  [plot] no user counts to plot")
        return
    vals = list(user_counts.values())
    # For readability, we can clip to, say, 50 purchases
    clipped = [min(v, 50) for v in vals]

    plt.figure()
    plt.hist(clipped, bins=50)
    plt.xlabel("Purchases per user (clipped at 50)")
    plt.ylabel("Number of users")
    plt.title(title)
    plt.yscale("log")  # user counts are very skewed
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()
    print(f"  [plot] saved {out_path}")


# Main per-category processing

def process_category(
    category: str,
    raw_dir: Path,
    processed_dir: Path,
    cleanup_raw: bool,
    cleanup_processed: str,
    allow_download: bool,
    repo_root: Path,
) -> None:
    """
    Process a single category with granular skipping and Drive-backed locking:
      - Skip whole category if ALL expected outputs exist (locally or via Drive).
      - Otherwise, skip individual steps if their outputs exist.
      - Upload processed outputs + lockfiles to Drive.
      - Optionally delete raw gz and/or processed outputs locally.
    """
    print(f"\n=== Category: {category} ===")

    # ------------- Lock (per-script, per-category, Drive-aware) -------------
    lock_dir = repo_root / "data" / "locks" / "01_build_user_purchase_counts"
    lock_path = lock_dir / f"{category}.lock"

    # Try to hydrate remote lock from Drive (if it exists) before checking
    rel_lock = str(lock_path.relative_to(repo_root))
    try:
        ensure_local_path(rel_lock)
    except Exception:
        pass

    lock_dir.mkdir(parents=True, exist_ok=True)

    if lock_path.exists():
        print(
            f"  [lock] Detected existing lock for {category} at {lock_path}. "
            "Skipping this category."
        )
        return

    with open(lock_path, "w", encoding="utf-8") as lf:
        lf.write("locked\n")
    try:
        upload_to_drive(lock_path)
    except Exception as e:
        print(f"  [lock] WARNING: failed to upload lock to Drive: {e}")

    try:
        # Paths
        review_url = REVIEW_URL_TEMPLATE.format(category=category)
        meta_url = META_URL_TEMPLATE.format(category=category)

        review_gz_path = raw_dir / "reviews" / f"{category}.jsonl.gz"
        meta_gz_path = raw_dir / "meta" / f"meta_{category}.jsonl.gz"

        cat_proc_dir = processed_dir / category
        cat_proc_dir.mkdir(parents=True, exist_ok=True)

        # Expected processed outputs
        user_counts_path = cat_proc_dir / f"user_counts_{category}.parquet"
        review_stats_path = cat_proc_dir / f"review_stats_{category}.json"
        rating_hist_png = cat_proc_dir / f"rating_hist_{category}.png"
        helpful_hist_png = cat_proc_dir / f"helpful_votes_hist_{category}.png"
        user_purchases_png = cat_proc_dir / f"user_purchases_hist_{category}.png"
        meta_stats_path = cat_proc_dir / f"meta_stats_{category}.json"

        expected_outputs = [
            user_counts_path,
            review_stats_path,
            rating_hist_png,
            helpful_hist_png,
            user_purchases_png,
            meta_stats_path,
        ]

        # Try to hydrate processed outputs from Drive
        ensure_outputs_from_drive(expected_outputs, repo_root)

        # --- Category-level skip if everything is already there ---
        if all(p.exists() for p in expected_outputs):
            print("  [skip] all processed outputs exist for this category; skipping heavy work.")
            if cleanup_raw:
                for p in (review_gz_path, meta_gz_path):
                    if p.exists():
                        print(f"  [cleanup] removing {p}")
                        p.unlink()
            print(f"=== Done category (skipped): {category} ===")
            return

        # Always ensure gz files are present before any step that might need them
        ensure_raw_gzip_or_download(
            review_gz_path, review_url, allow_download, repo_root
        )
        ensure_raw_gzip_or_download(
            meta_gz_path, meta_url, allow_download, repo_root
        )

        # -------------------------------------------------------------------
        # Step 1: ensure review_stats + user_counts exist
        # -------------------------------------------------------------------
        user_counts_df = None
        rating_hist = None
        helpful_hist = None

        if user_counts_path.exists() and review_stats_path.exists():
            print("  [skip] user_counts + review_stats already exist; loading from disk.")
            # Load from existing artifacts
            user_counts_df = pd.read_parquet(user_counts_path)
            with open(review_stats_path, "r", encoding="utf-8") as f:
                stats = json.load(f)
            rating_hist = Counter(
                {float(k): v for k, v in stats["rating_hist"].items()}
            )
            helpful_hist = Counter(
                {int(k): v for k, v in stats["helpful_hist"].items()}
            )
        else:
            print("  [run] computing review_stats + user_counts from gz.")
            # (Re)compute from gz, with re-download protection for truncated file
            try:
                review_stats_raw = process_review_file(
                    review_gz_path, category
                )
            except EOFError:
                print(
                    f"  [warn] Detected truncated gzip for {category}, re-downloading..."
                )
                download_if_needed(review_url, review_gz_path, force=True)
                review_stats_raw = process_review_file(
                    review_gz_path, category
                )

            user_counts = review_stats_raw["user_counts"]
            rating_hist = review_stats_raw["rating_hist"]
            helpful_hist = review_stats_raw["helpful_hist"]

            # Build DataFrame for ALL users, no filtering
            user_counts_rows = [
                {"user_id": uid, "num_purchases": cnt, "category": category}
                for uid, cnt in user_counts.items()
            ]
            user_counts_df = pd.DataFrame(user_counts_rows)
            user_counts_df.to_parquet(user_counts_path, index=False)
            print(f"  [save] user_counts -> {user_counts_path}")

            # Save review stats JSON
            review_stats_out = {
                "category": category,
                "n_reviews": review_stats_raw["n_reviews"],
                "n_users": len(user_counts),
                "n_verified": review_stats_raw["n_verified"],
                "rating_hist": dict(rating_hist),
                "helpful_hist": dict(helpful_hist),
            }
            with open(review_stats_path, "w", encoding="utf-8") as f:
                json.dump(review_stats_out, f, indent=2)
            print(f"  [save] review_stats -> {review_stats_path}")

        # -------------------------------------------------------------------
        # Step 2: ensure plots exist (rating, helpful, user_purchases)
        # -------------------------------------------------------------------
        if not rating_hist_png.exists():
            save_rating_hist_plot(
                rating_hist,
                rating_hist_png,
                title=f"Rating distribution ({category})",
            )
        else:
            print(f"  [skip] rating hist plot already exists: {rating_hist_png}")

        if not helpful_hist_png.exists():
            save_helpful_hist_plot(
                helpful_hist,
                helpful_hist_png,
                title=f"Helpful votes distribution ({category})",
            )
        else:
            print(
                f"  [skip] helpful votes plot already exists: {helpful_hist_png}"
            )

        if not user_purchases_png.exists():
            # user_counts_df is guaranteed to be loaded at this point
            save_user_purchases_hist_plot(
                dict(zip(user_counts_df["user_id"], user_counts_df["num_purchases"])),
                user_purchases_png,
                title=f"Purchases per user ({category})",
            )
        else:
            print(
                f"  [skip] user purchases plot already exists: {user_purchases_png}"
            )

        # -------------------------------------------------------------------
        # Step 3: ensure meta_stats exist
        # -------------------------------------------------------------------
        if meta_stats_path.exists():
            print(f"  [skip] meta_stats already exist: {meta_stats_path}")
        else:
            print("  [run] computing meta_stats from gz.")
            try:
                meta_stats_raw = process_meta_file(meta_gz_path)
            except EOFError:
                print(
                    f"  [warn] Detected truncated meta gzip for {category}, re-downloading..."
                )
                download_if_needed(meta_url, meta_gz_path, force=True)
                meta_stats_raw = process_meta_file(meta_gz_path)

            meta_stats_out = {
                "category": category,
                "n_items": meta_stats_raw["n_items"],
                "rating_number_hist": dict(
                    meta_stats_raw["rating_number_hist"]
                ),
                "price_count": meta_stats_raw["price_count"],
                "price_mean": (
                    meta_stats_raw["price_sum"]
                    / meta_stats_raw["price_count"]
                    if meta_stats_raw["price_count"] > 0
                    else None
                ),
            }
            with open(meta_stats_path, "w", encoding="utf-8") as f:
                json.dump(meta_stats_out, f, indent=2)
            print(f"  [save] meta_stats -> {meta_stats_path}")

        # -------------------------------------------------------------------
        # Upload processed artifacts to Drive
        # -------------------------------------------------------------------
        upload_targets = [
            user_counts_path,
            review_stats_path,
            rating_hist_png,
            helpful_hist_png,
            user_purchases_png,
            meta_stats_path,
        ]
        print("  [drive] uploading processed outputs to Drive...")
        for p in upload_targets:
            try:
                upload_to_drive(p)
            except Exception as e:
                print(f"  [drive] WARNING: failed to upload {p}: {e}")

        # -------------------------------------------------------------------
        # Cleanup processed outputs (optional)
        # -------------------------------------------------------------------
        if cleanup_processed != "none":
            for p in upload_targets:
                if cleanup_processed == "parquet" and p.suffix != ".parquet":
                    continue
                # 'all' -> delete everything in upload_targets
                try:
                    if p.exists():
                        print(f"  [cleanup-processed] removing {p}")
                        p.unlink()
                except OSError as e:
                    print(f"  [cleanup-processed] WARNING: failed to remove {p}: {e}")

        # -------------------------------------------------------------------
        # Cleanup raw gz files if requested
        # -------------------------------------------------------------------
        if cleanup_raw:
            for p in (review_gz_path, meta_gz_path):
                if p.exists():
                    print(f"  [cleanup-raw] removing {p}")
                    p.unlink()

        print(f"=== Done category: {category} ===")

    finally:
        # Always try to release lock
        if lock_path.exists():
            try:
                lock_path.unlink()
            except OSError:
                pass


# Category selection

def read_all_categories_from_file(path: Path) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Preprocess Amazon-Reviews-2023 categories (user counts + EDA)."
    )
    parser.add_argument(
        "--categories",
        nargs="*",
        help="List of category names to process (default: all from all_categories.txt).",
    )
    parser.add_argument(
        "--categories-file",
        type=str,
        default=None,
        help="Path to all_categories.txt (default: data/raw/all_categories.txt under repo).",
    )
    parser.add_argument(
        "--no-cleanup-raw",
        action="store_true",
        help="If set, keep the downloaded .jsonl.gz files instead of deleting them.",
    )
    parser.add_argument(
        "--no-cleanup",
        action="store_true",
        help="Alias for --no-cleanup-raw (backwards compatibility).",
    )
    parser.add_argument(
        "--cleanup-processed",
        choices=["none", "parquet", "all"],
        default="parquet",
        help=(
            "How to clean up processed outputs after uploading to Drive. "
            "'none' = keep everything; "
            "'parquet' = remove .parquet only (default); "
            "'all' = remove parquet, JSON, and PNGs."
        ),
    )
    parser.add_argument(
        "--no-download",
        action="store_true",
        help="Don't attempt to download missing raw files from UCSD; fail if not present.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    repo_root = get_repo_root()
    raw_dir = repo_root / "data" / "raw"
    processed_dir = repo_root / "data" / "processed"

    cleanup_raw = not (args.no_cleanup_raw or args.no_cleanup)
    cleanup_processed = args.cleanup_processed
    allow_download = not args.no_download

    # Always resync registry at the start so Drive state is fresh
    resync_registry()

    # Determine categories
    if args.categories:
        categories = args.categories
        print(f"Using categories from CLI: {categories}")
    else:
        if args.categories_file:
            cat_file = Path(args.categories_file)
            if not cat_file.is_absolute():
                cat_file = repo_root / cat_file
            if not cat_file.exists():
                rel = str(cat_file.relative_to(repo_root))
                print(
                    f"[info] categories file not local; trying Drive for {rel}"
                )
                cat_file = ensure_local_path(rel)
        else:
            # Use registry entry raw.all_categories.txt by default
            cat_file = ensure_local("raw", "all_categories.txt")

        categories = read_all_categories_from_file(cat_file)
        print(f"No categories specified; using all from {cat_file}")
        print(f"{len(categories)} categories: {categories}")

    for cat in categories:
        try:
            process_category(
                cat,
                raw_dir=raw_dir,
                processed_dir=processed_dir,
                cleanup_raw=cleanup_raw,
                cleanup_processed=cleanup_processed,
                allow_download=allow_download,
                repo_root=repo_root,
            )
        except Exception as e:
            print(f"  [error] processing {cat}: {e}")


if __name__ == "__main__":
    main()

# examples:
# python common_scripts/01_build_user_purchase_counts.py --categories All_Beauty Toys_and_Games
# python common_scripts/01_build_user_purchase_counts.py --no-cleanup-raw
# python common_scripts/01_build_user_purchase_counts.py --cleanup-processed all

In [ ]:
"""
Script 2: Global User Collation + User Importance Scoring
Updated per requirements:
- Save histograms for:
    * total purchases per user
    * distinct categories per user
- Extract top users that satisfy:
    * importance >= 95th percentile
    * distinct_categories >= 3
    * total_purchases >= 3
"""

import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

BASE_DIR = pathlib.Path("data/processed")
OUT_DIR = pathlib.Path("data/global")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Compute entropy from category counts
# ---------------------------------------------------------
def compute_entropy(counts: np.ndarray) -> float:
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    p = p[p > 0]  # avoid log(0)
    return -(p * np.log(p)).sum()


# ---------------------------------------------------------
# Load all per-category user_count tables
# ---------------------------------------------------------
def load_all_user_counts():
    files = list(BASE_DIR.glob("*/user_counts_*.parquet"))

    dfs = []
    for f in tqdm(files, desc="Loading user_counts"):
        dfs.append(pd.read_parquet(f))

    return pd.concat(dfs, ignore_index=True)


# ---------------------------------------------------------
# Aggregate into global user‐category matrix
# ---------------------------------------------------------
def aggregate_user_data(df:pd.DataFrame):
    pivot = df.pivot_table(
        index="user_id",
        columns="category",
        values="num_purchases",
        aggfunc="sum",
        fill_value=0
    )

    pivot["total_purchases"] = pivot.sum(axis=1)
    pivot["distinct_categories"] = (pivot.drop(columns="total_purchases") > 0).sum(axis=1)
    return pivot


# ---------------------------------------------------------
# Compute user diversity and importance
# ---------------------------------------------------------
def compute_user_importance(df:pd.DataFrame):
    category_cols = [c for c in df.columns if c not in ["total_purchases", "distinct_categories"]]

    df["entropy"] = df[category_cols].apply(lambda row: compute_entropy(row.values), axis=1)

    max_entropy = np.log(len(category_cols))
    df["norm_entropy"] = df["entropy"] / max_entropy

    df["importance"] = df["total_purchases"] * (1 + df["norm_entropy"])

    return df


# ---------------------------------------------------------
# Extract top users with constraints
# ---------------------------------------------------------
def extract_top_users(df:pd.DataFrame, percentile=0.95):
    importance_threshold = df["importance"].quantile(percentile)

    filtered = df[
        (df["importance"] >= importance_threshold) &
        (df["total_purchases"] >= 3) &
        (df["distinct_categories"] >= 3)
    ]

    return filtered


# ---------------------------------------------------------
# Plot histograms and save to disk
# ---------------------------------------------------------
def save_histograms(df:pd.DataFrame):
    # Total purchases hist
    plt.figure(figsize=(8, 5))
    df["total_purchases"].clip(upper=50).hist(bins=50)
    plt.title("User Total Purchase Counts (clipped at 50)")
    plt.xlabel("Total Purchases")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "user_total_purchases_hist.png")
    plt.close()

    # Distinct categories hist
    plt.figure(figsize=(8, 5))
    df["distinct_categories"].hist(bins=50)
    plt.title("Distinct Categories per User")
    plt.xlabel("Num Categories")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "user_distinct_categories_hist.png")
    plt.close()


# ---------------------------------------------------------
# Main
# ---------------------------------------------------------
def main():
    print(">>> Loading per-category user data...")
    raw = load_all_user_counts()

    print(">>> Aggregating global user table...")
    user_df = aggregate_user_data(raw)

    print(">>> Generating histograms...")
    save_histograms(user_df)

    print(">>> Computing user importance...")
    user_df = compute_user_importance(user_df)

    print(">>> Extracting top users (with min categories=3, min purchases=3)...")
    top_users = extract_top_users(user_df, percentile=0.95)

    top_out = OUT_DIR / "top_users.parquet"
    top_users.to_parquet(top_out)
    print(f"Saved top users → {top_out}")

    print("\n[ DONE ]\n")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
"""
Extract per-review, per-user, and per-item features for top users (per-category).

For each category (or a list of categories):
  - Load global top_users (importance scores) from data/global/top_users.parquet
    via the data registry / Google Drive (unless overridden via CLI).
  - Stream the category review .jsonl.gz and keep only reviews by top users.
  - Produce per-review feature table:
      data/processed/<Category>/top_user_reviews_<Category>.parquet
  - Produce per-user aggregated features:
      data/processed/<Category>/top_user_features_<Category>.parquet
  - Produce per-item aggregated features (from top users only):
      data/processed/<Category>/top_item_features_<Category>.parquet
  - Produce EDA outputs:
      top_user_review_stats_<Category>.json
      top_users_rating_hist_<Category>.png
      top_users_helpful_hist_<Category>.png

Step-based skipping & Drive integration:
  - Resync registry at start of script.
  - For each category:
      - Use Drive-backed lockfiles under data/locks/03_user_features/.
      - Try to hydrate processed outputs from Drive to enable skipping.
      - Skip category if all outputs exist.
      - For needed steps, ensure raw gz exist via Drive or UCSD download.
      - Upload processed outputs + lockfiles to Drive.
      - Optional cleanup of raw gz and/or processed outputs locally.
"""

import argparse
import gzip
import json
import time
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import matplotlib.pyplot as plt
import pandas as pd

# Make sure we can import siblings (data_io.py) no matter where we run it from
THIS_DIR = Path(__file__).resolve().parent
if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

from data_io import (  # noqa: E402
    ensure_local,
    ensure_local_path,
    resync_registry,
    upload_to_drive,
    delete_remote_by_rel_path,
    remote_file_exists_by_rel_path,
)


def get_repo_root() -> Path:
    return Path(__file__).resolve().parents[1]


# --------------------------------------------------------------------
# Plot helpers
# --------------------------------------------------------------------
def save_rating_hist_plot(
    rating_hist: Counter, out_path: Path, title: str
) -> None:
    if not rating_hist:
        print("  [plot] no ratings to plot")
        return
    items = sorted(rating_hist.items())
    xs = [k for k, _ in items]
    ys = [v for _, v in items]

    plt.figure()
    plt.bar(xs, ys)
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()
    print(f"  [plot] saved {out_path}")


def save_helpful_hist_plot(
    helpful_hist: Counter, out_path: Path, title: str
) -> None:
    if not helpful_hist:
        print("  [plot] no helpful_votes to plot")
        return
    items = sorted(helpful_hist.items())
    xs = [k for k, _ in items]
    ys = [v for _, v in items]

    plt.figure()
    plt.bar(xs, ys)
    plt.xlabel("Helpful votes (clipped)")
    plt.ylabel("Count")
    plt.title(title)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.close()
    print(f"  [plot] saved {out_path}")


# --------------------------------------------------------------------
# Raw data URLs
# --------------------------------------------------------------------
REVIEW_URL_TEMPLATE = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/"
    "review_categories/{category}.jsonl.gz"
)

META_URL_TEMPLATE = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/"
    "meta_categories/meta_{category}.jsonl.gz"
)


def download_if_needed(url: str, dest: Path, force: bool = False) -> None:
    # Keep a small download helper to mirror 01 script behavior
    import requests

    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not force:
        print(f"  [exists] {dest}")
        return

    print(f"  [download{' (force)' if force else ''}] {url} -> {dest}")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
    print("  [ok] download complete")


# --------------------------------------------------------------------
# CLI handling
# --------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Extract top-user filtered features per category."
    )
    parser.add_argument(
        "--categories",
        nargs="*",
        help="List of category names to process (default: all from data/raw/all_categories.txt).",
    )
    parser.add_argument(
        "--categories-file",
        type=str,
        default=None,
        help=(
            "Path to all_categories.txt (default: data/raw/all_categories.txt under repo; "
            "will be fetched from Drive via registry if needed)."
        ),
    )
    parser.add_argument(
        "--top-users",
        type=str,
        default=None,
        help=(
            "Path to top_users.parquet. "
            "If omitted, will use data registry entry processed.top_users "
            "and download from Drive if needed."
        ),
    )
    parser.add_argument(
        "--no-download",
        action="store_true",
        help="Don't attempt to download missing raw files from UCSD; fail if not present.",
    )
    parser.add_argument(
        "--no-cleanup-raw",
        action="store_true",
        help="If set, keep the downloaded .jsonl.gz files instead of deleting them.",
    )
    parser.add_argument(
        "--no-cleanup",
        action="store_true",
        help="Alias for --no-cleanup-raw (backwards compatibility).",
    )
    parser.add_argument(
        "--cleanup-processed",
        choices=["none", "parquet", "all"],
        default="parquet",
        help=(
            "How to clean up processed outputs after uploading to Drive. "
            "'none' = keep everything; "
            "'parquet' = remove .parquet only (default); "
            "'all' = remove parquet, JSON, and PNGs."
        ),
    )
    parser.add_argument(
        "--progress-interval",
        type=int,
        default=1_000_000,
        help=(
            "Print in-category parse progress every N lines while streaming "
            "the review .jsonl.gz (default: 1,000,000)."
        ),
    )
    return parser.parse_args()


def read_all_categories_from_file(path: Path) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


# --------------------------------------------------------------------
# Helpers for top_users and meta
# --------------------------------------------------------------------
def load_top_users(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    # ensure we have a `user_id` column
    if "user_id" in df.columns:
        return df
    df = df.reset_index()
    if "user_id" not in df.columns:
        df = df.rename(columns={df.columns[0]: "user_id"})
    return df


def load_item_meta(meta_gz: Path, category: str) -> Dict[str, Dict]:
    """
    Load item-level metadata from meta_<Category>.jsonl.gz into a dict:
      parent_asin -> {item_avg_rating, item_categories}
    """
    item_meta: Dict[str, Dict] = {}
    if not meta_gz.exists():
        print(
            f"  [meta] no meta file found for {category} at {meta_gz}; "
            "continuing without item meta"
        )
        return item_meta

    print(
        f"  [meta] loading meta from {meta_gz} "
        "(to attach item-level avg rating + categories)"
    )

    first_bad = True
    with gzip.open(meta_gz, "rt", encoding="utf-8") as mf:
        for mline in mf:
            mline = mline.strip()
            if not mline:
                continue
            try:
                mobj = json.loads(mline)
            except json.JSONDecodeError as e:
                if first_bad:
                    print(
                        f"  [meta] WARNING: skipping malformed JSON line in "
                        f"{category}: {e}"
                    )
                    first_bad = False
                continue

            a = (
                mobj.get("parent_asin")
                or mobj.get("asin")
                or mobj.get("asin")
            )
            if not a:
                continue

            # possible fields for item average rating
            item_avg = None
            for k in ("avg_rating", "average_rating", "rating"):
                v = mobj.get(k)
                if v is not None:
                    try:
                        item_avg = float(v)
                        break
                    except Exception:
                        pass

            # Categories can be nested lists; try to flatten
            cats = None
            raw_cats = mobj.get("categories") or mobj.get("category")
            if raw_cats:
                if isinstance(raw_cats, list):
                    flat = []
                    for el in raw_cats:
                        if isinstance(el, list):
                            flat.extend(
                                [str(x) for x in el if x is not None]
                            )
                        else:
                            flat.append(str(el))
                    cats = list(dict.fromkeys(flat))
                else:
                    cats = [str(raw_cats)]

            item_meta[str(a)] = {
                "item_avg_rating": item_avg,
                "item_categories": cats,
            }

    return item_meta


# --------------------------------------------------------------------
# Raw gz helper (Drive + UCSD)
# --------------------------------------------------------------------
def ensure_raw_gzip_or_download(
    path: Path, url: str, allow_download: bool, repo_root: Path
) -> None:
    """
    Ensure a raw gzip exists locally by:
      1) Checking local path
      2) Trying Drive via ensure_local_path (using relative path)
      3) Falling back to direct HTTP download from UCSD if allowed
    """
    if path.exists():
        return

    rel = str(path.relative_to(repo_root))
    try:
        print(f"  [data_io] trying Drive for {rel}")
        ensure_local_path(rel)
        if path.exists():
            return
    except Exception:
        # No registry entry or download failed -> fall back
        pass

    if allow_download:
        download_if_needed(url, path, force=False)
    else:
        raise FileNotFoundError(
            f"Missing {path} and --no-download is set; "
            "no Drive entry or HTTP download attempted."
        )


def ensure_outputs_from_drive(paths: List[Path], repo_root: Path) -> None:
    """
    For each expected output, if it's missing locally but exists in Drive
    (according to registry), pull it down so we can skip work.
    """
    for p in paths:
        if p.exists():
            continue
        rel = str(p.relative_to(repo_root))
        try:
            ensure_local_path(rel)
        except Exception:
            # No registry entry or download failed – ignore.
            pass


def count_gzip_lines(path: Path, category: str) -> int:
    """
    One-time pre-pass to count total lines in a gzip JSONL file so that
    we can report parse progress as N/total and estimate ETA.
    """
    print(f"  [scan] counting lines in {path} for {category}...")
    start = time.time()
    n = 0
    # Use errors='ignore' to be robust to any stray bytes
    with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
        for _ in f:
            n += 1
    elapsed = time.time() - start
    print(
        f"  [scan] {category}: total_lines={n:,} (took {elapsed:.1f}s)"
    )
    return n


# --------------------------------------------------------------------
# Core per-category processing
# --------------------------------------------------------------------
def parse_reviews_for_top_users(
    review_gz: Path,
    top_users_set,
    item_meta: Dict[str, Dict],
    category: str,
    progress_interval: int,
    total_lines: Optional[int],
    out_reviews_parquet: Path,
    parquet_batch_size: int = 500_000,
) -> Tuple[Counter, Counter, int, int]:
    """
    Stream the review JSONL.gz, filter to top users, attach item meta,
    and write per-review data directly to Parquet in batches.

    Returns:
      rating_hist: Counter of rating -> count
      helpful_hist: Counter of clipped helpful votes -> count
      n_users_with_reviews: number of unique users with at least one review
      n_reviews_kept: total number of kept reviews

    Notes:
      - We do NOT materialize a full reviews DataFrame in memory anymore.
      - Per-review data is written incrementally to `out_reviews_parquet`.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    # If a previous file exists and we're re-running, overwrite it
    if out_reviews_parquet.exists():
        out_reviews_parquet.unlink()

    rating_hist = Counter()
    helpful_hist = Counter()
    seen_users = set()

    n_lines = 0
    n_kept = 0
    start_time = time.time()

    rows_batch: List[Dict] = []
    writer = None

    def flush_batch() -> None:
        nonlocal writer, rows_batch
        if not rows_batch:
            return
        # Convert list-of-dicts to column-wise dict for pyarrow
        keys = rows_batch[0].keys()
        cols = {k: [row[k] for row in rows_batch] for k in keys}
        table = pa.Table.from_pydict(cols)
        if writer is None:
            writer = pq.ParquetWriter(str(out_reviews_parquet), table.schema)
        writer.write_table(table)
        rows_batch = []

    with gzip.open(review_gz, "rt", encoding="utf-8") as f:
        for line in f:
            n_lines += 1
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            user_id = obj.get("user_id")
            if user_id is None:
                continue
            user_id = str(user_id)
            if user_id not in top_users_set:
                continue

            product_id = (
                obj.get("asin")
                or obj.get("product_id")
                or obj.get("item_id")
                or obj.get("id")
            )
            rating = obj.get("rating")

            hv = obj.get("helpful_votes", obj.get("helpful_vote", 0))
            try:
                hv = int(hv)
            except (TypeError, ValueError):
                hv = 0
            hv_clipped = min(hv, 50)

            unix_time = (
                obj.get("timestamp")
                or obj.get("unixReviewTime")
                or obj.get("unix_review_time")
                or obj.get("unix_time")
            )

            verified = (
                obj.get("verified_purchase") or obj.get("verified") or False
            )

            im = (
                item_meta.get(str(product_id), {})
                if product_id is not None
                else {}
            )

            row = {
                "user_id": user_id,
                "product_id": product_id,
                "unixReviewTime": int(unix_time)
                if unix_time is not None
                else None,
                "rating": float(rating) if rating is not None else None,
                "helpful_votes": hv,
                "helpful_votes_clipped": hv_clipped,
                "verified_purchase": bool(verified),
                "item_avg_rating": im.get("item_avg_rating"),
                "item_categories": im.get("item_categories"),
            }

            rows_batch.append(row)
            seen_users.add(user_id)
            n_kept += 1

            # Flush Parquet batch if needed to keep memory bounded
            if len(rows_batch) >= parquet_batch_size:
                flush_batch()

            # Progress / ETA logging
            if progress_interval > 0 and n_lines % progress_interval == 0:
                elapsed = time.time() - start_time
                rate = n_lines / elapsed if elapsed > 0 else 0.0

                if total_lines:
                    frac = min(n_lines / total_lines, 1.0)
                    pct = frac * 100.0
                    eta_str = "unknown"
                    if rate > 0:
                        remaining = max(total_lines - n_lines, 0)
                        eta_sec = remaining / rate
                        mins = int(eta_sec // 60)
                        secs = int(eta_sec % 60)
                        eta_str = f"{mins}m {secs}s"
                    print(
                        f"    [parse] {category}: "
                        f"{n_lines:,}/{total_lines:,} lines "
                        f"({pct:5.1f}%), kept={n_kept:,}, "
                        f"users_with_reviews={len(seen_users):,}, "
                        f"rate={rate:,.0f} lines/s, ETA={eta_str}"
                    )
                else:
                    print(
                        f"    [parse] {category}: "
                        f"lines_seen={n_lines:,}, kept={n_kept:,}, "
                        f"users_with_reviews={len(seen_users):,}"
                    )

            # Histograms
            try:
                if rating is not None:
                    rating_hist[float(rating)] += 1
            except (TypeError, ValueError):
                pass
            helpful_hist[hv_clipped] += 1

    # Flush any remaining rows
    flush_batch()
    if writer is not None:
        writer.close()
    else:
        # No rows kept at all → create an empty parquet with expected columns
        empty_df = pd.DataFrame(
            columns=[
                "user_id",
                "product_id",
                "unixReviewTime",
                "rating",
                "helpful_votes",
                "helpful_votes_clipped",
                "verified_purchase",
                "item_avg_rating",
                "item_categories",
            ]
        )
        empty_df.to_parquet(out_reviews_parquet, index=False)

    if total_lines:
        print(
            f"  [done] filtered reviews rows={n_kept:,}, "
            f"users_with_reviews={len(seen_users):,}, "
            f"lines_seen={n_lines:,}/{total_lines:,}"
        )
    else:
        print(
            f"  [done] filtered reviews rows={n_kept:,}, "
            f"users_with_reviews={len(seen_users):,}, "
            f"lines_seen={n_lines:,}"
        )

    return rating_hist, helpful_hist, len(seen_users), n_kept


def process_category(
    category: str,
    raw_dir: Path,
    processed_dir: Path,
    top_users_df: pd.DataFrame,
    top_users_set,
    allow_download: bool,
    cleanup_raw: bool,
    cleanup_processed: str,
    repo_root: Path,
    progress_interval: int,
):
    print(f"\n=== Category (top-users filtered): {category} ===")

    # Paths and expected outputs (used for lock + skip decisions)
    review_gz = raw_dir / "reviews" / f"{category}.jsonl.gz"
    meta_gz = raw_dir / "meta" / f"meta_{category}.jsonl.gz"
    review_url = REVIEW_URL_TEMPLATE.format(category=category)
    meta_url = META_URL_TEMPLATE.format(category=category)

    cat_proc_dir = processed_dir / category
    cat_proc_dir.mkdir(parents=True, exist_ok=True)

    out_reviews_parquet = cat_proc_dir / f"top_user_reviews_{category}.parquet"
    out_user_features = cat_proc_dir / f"top_user_features_{category}.parquet"
    out_item_features = cat_proc_dir / f"top_item_features_{category}.parquet"
    out_stats_json = cat_proc_dir / f"top_user_review_stats_{category}.json"
    rating_png = cat_proc_dir / f"top_users_rating_hist_{category}.png"
    helpful_png = cat_proc_dir / f"top_users_helpful_hist_{category}.png"

    expected_outputs = [
        out_reviews_parquet,
        out_user_features,
        out_item_features,
        out_stats_json,
        rating_png,
        helpful_png,
    ]

    # ----------------- Locking (Drive-aware, per-category) -------------------
    lock_dir = repo_root / "data" / "locks" / "03_user_features"
    lock_dir.mkdir(parents=True, exist_ok=True)
    lock_path = lock_dir / f"{category}.lock"
    rel_lock = str(lock_path.relative_to(repo_root))

    # If a remote lock exists, hydrate it locally (if needed)
    try:
        if remote_file_exists_by_rel_path(rel_lock) and not lock_path.exists():
            ensure_local_path(rel_lock)
    except Exception:
        # If Drive is temporarily unavailable, we'll fall back to local state
        pass

    if lock_path.exists():
        print(
            f"  [lock] Detected existing lock for {category} at {lock_path}. "
            "Checking whether category is fully completed..."
        )
        # Pull any missing outputs from Drive so we can make a real decision
        ensure_outputs_from_drive(expected_outputs, repo_root)

        if all(p.exists() for p in expected_outputs):
            # Category is fully done; clear lock and skip
            print(
                "  [lock] All expected outputs exist locally; "
                "treating category as complete and clearing lock."
            )
            try:
                lock_path.unlink()
            except OSError:
                pass
            try:
                delete_remote_by_rel_path(rel_lock)
            except Exception as e:
                print(f"  [lock] WARNING: failed to delete remote lock: {e}")
            print(f"=== Done category (completed; lock cleared): {category} ===")
            return
        else:
            # Stale lock: outputs missing, so we take over
            print(
                "  [lock] Lock exists but outputs are incomplete; "
                "assuming stale lock and taking over."
            )
            try:
                lock_path.unlink()
            except OSError:
                pass
            try:
                delete_remote_by_rel_path(rel_lock)
            except Exception as e:
                print(f"  [lock] WARNING: failed to delete remote lock: {e}")

    # If we get here, either there was no lock or we just cleared a stale one.
    # Create a fresh lock for this run and push it to Drive.
    with open(lock_path, "w", encoding="utf-8") as lf:
        lf.write("locked\n")
    try:
        upload_to_drive(lock_path)
    except Exception as e:
        print(f"  [lock] WARNING: failed to upload lock to Drive: {e}")

    try:
        review_gz = raw_dir / "reviews" / f"{category}.jsonl.gz"
        meta_gz = raw_dir / "meta" / f"meta_{category}.jsonl.gz"
        review_url = REVIEW_URL_TEMPLATE.format(category=category)
        meta_url = META_URL_TEMPLATE.format(category=category)

        cat_proc_dir = processed_dir / category
        cat_proc_dir.mkdir(parents=True, exist_ok=True)

        # Outputs
        out_reviews_parquet = cat_proc_dir / f"top_user_reviews_{category}.parquet"
        out_user_features = cat_proc_dir / f"top_user_features_{category}.parquet"
        out_item_features = cat_proc_dir / f"top_item_features_{category}.parquet"
        out_stats_json = cat_proc_dir / f"top_user_review_stats_{category}.json"
        rating_png = cat_proc_dir / f"top_users_rating_hist_{category}.png"
        helpful_png = cat_proc_dir / f"top_users_helpful_hist_{category}.png"

        expected_outputs = [
            out_reviews_parquet,
            out_user_features,
            out_item_features,
            out_stats_json,
            rating_png,
            helpful_png,
        ]

        # Try to pull processed outputs from Drive if they exist there
        ensure_outputs_from_drive(expected_outputs, repo_root)

        # Category-level skip if EVERYTHING exists
        if all(p.exists() for p in expected_outputs):
            print("  [skip] all processed outputs exist for this category")
            # Optional cleanup of raw gz if requested
            if cleanup_raw:
                for p in (review_gz, meta_gz):
                    if p.exists():
                        print(f"  [cleanup-raw] removing {p}")
                        p.unlink()
            print(f"=== Done category (skipped): {category} ===")
            return

        # Determine which steps we actually need
        need_step1 = not (
            out_reviews_parquet.exists() and out_stats_json.exists()
        )
        need_step2 = not out_user_features.exists()
        need_step3 = not out_item_features.exists()
        need_step4 = not (rating_png.exists() and helpful_png.exists())

        total_lines: Optional[int] = None

        # We need the review gzip if we are going to re-parse
        if need_step1:
            ensure_raw_gzip_or_download(
                review_gz, review_url, allow_download, repo_root
            )
            # One-time pre-pass to get total number of lines for progress + ETA
            total_lines = count_gzip_lines(review_gz, category)

        # We need meta if we are going to parse or create item-level aggregates
        need_meta = need_step1 or need_step3
        item_meta: Dict[str, Dict] = {}

        if need_meta:
            ensure_raw_gzip_or_download(
                meta_gz, meta_url, allow_download, repo_root
            )
            if meta_gz.exists():
                try:
                    item_meta = load_item_meta(meta_gz, category)
                except EOFError:
                    print(
                        f"  [warn] truncated meta gzip for {category}; "
                        "re-downloading and retrying..."
                    )
                    download_if_needed(meta_url, meta_gz, force=True)
                    item_meta = load_item_meta(meta_gz, category)

        print(
            f"  [info] top_users provided: {len(top_users_set)} users; filtering reviews..."
        )

        # ------------------------------------------------------------------
        # Step 1: filtered reviews + stats (streaming parser)
        # ------------------------------------------------------------------
        if not need_step1:
            print("  [skip] filtered reviews + stats already exist; loading from disk.")
            with open(out_stats_json, "r", encoding="utf-8") as f:
                stats = json.load(f)
            rating_hist = Counter(
                {float(k): v for k, v in stats.get("rating_hist", {}).items()}
            )
            helpful_hist = Counter(
                {int(k): v for k, v in stats.get("helpful_hist", {}).items()}
            )
            n_reviews = int(stats.get("n_reviews", 0))
            n_users = int(stats.get("n_users", 0))
        else:
            print("  [run] parsing review gz for top users (streaming)...")
            try:
                rating_hist, helpful_hist, n_users, n_reviews = (
                    parse_reviews_for_top_users(
                        review_gz,
                        top_users_set,
                        item_meta,
                        category,
                        progress_interval,
                        total_lines,
                        out_reviews_parquet,
                    )
                )
            except EOFError:
                print(
                    f"  [warn] truncated review gzip for {category}; "
                    "re-downloading and retrying..."
                )
                download_if_needed(review_url, review_gz, force=True)
                rating_hist, helpful_hist, n_users, n_reviews = (
                    parse_reviews_for_top_users(
                        review_gz,
                        top_users_set,
                        item_meta,
                        category,
                        progress_interval,
                        total_lines,
                        out_reviews_parquet,
                    )
                )

            stats_out = {
                "category": category,
                "n_reviews": int(n_reviews),
                "n_users": int(n_users),
                "rating_hist": dict(rating_hist),
                "helpful_hist": dict(helpful_hist),
            }
            with open(out_stats_json, "w", encoding="utf-8") as f:
                json.dump(stats_out, f, indent=2)
            print(f"  [save] stats -> {out_stats_json}")

        # If there are zero reviews, short-circuit steps 2–4
        if n_reviews == 0:
            print("  [warn] No reviews for top users in this category.")
            if not out_user_features.exists():
                pd.DataFrame(columns=["user_id"]).to_parquet(
                    out_user_features, index=False
                )
                print(
                    f"  [save] empty per-user features -> {out_user_features}"
                )
            if not out_item_features.exists():
                pd.DataFrame(columns=["product_id"]).to_parquet(
                    out_item_features, index=False
                )
                print(
                    f"  [save] empty per-item features -> {out_item_features}"
                )
            if need_step4:
                save_rating_hist_plot(
                    rating_hist,
                    rating_png,
                    title=f"Top users rating distribution ({category})",
                )
                save_helpful_hist_plot(
                    helpful_hist,
                    helpful_png,
                    title=f"Top users helpful votes ({category})",
                )
            # Cleanup raw gz if requested
            if cleanup_raw:
                for p in (review_gz, meta_gz):
                    if p.exists():
                        print(f"  [cleanup-raw] removing {p}")
                        p.unlink()
            print(f"=== Done category (no top-user reviews): {category} ===")
            return

        # ------------------------------------------------------------------
        # Load reviews DataFrame only if needed for groupbys
        # ------------------------------------------------------------------
        reviews_df: Optional[pd.DataFrame] = None
        if (need_step2 or need_step3) and n_reviews > 0:
            print("  [load] reading top-user reviews parquet for aggregation...")
            reviews_df = pd.read_parquet(out_reviews_parquet)

        # ------------------------------------------------------------------
        # Step 2: per-user aggregated features
        # ------------------------------------------------------------------
        if not need_step2:
            print(
                f"  [skip] per-user features already exist: {out_user_features}"
            )
        else:
            print("  [run] computing per-user aggregated features...")
            agg = (
                reviews_df.groupby("user_id")
                .agg(
                    num_purchases_in_category=("product_id", "count"),
                    avg_rating_in_category=("rating", "mean"),
                    avg_helpful_vote_in_category=("helpful_votes", "mean"),
                    avg_item_avg_rating=("item_avg_rating", "mean"),
                    first_review_time=("unixReviewTime", "min"),
                    last_review_time=("unixReviewTime", "max"),
                )
                .reset_index()
            )

            top_users_df_local = top_users_df.copy()
            if "user_id" not in top_users_df_local.columns:
                top_users_df_local = top_users_df_local.reset_index()

            top_users_df_local["user_id"] = top_users_df_local[
                "user_id"
            ].astype(str)
            agg["user_id"] = agg["user_id"].astype(str)

            sel = agg.merge(top_users_df_local, on="user_id", how="left")

            sel.to_parquet(out_user_features, index=False)
            print(f"  [save] per-user features -> {out_user_features}")

        # ------------------------------------------------------------------
        # Step 3: per-item aggregated features
        # ------------------------------------------------------------------
        if not need_step3:
            print(
                f"  [skip] per-item features already exist: {out_item_features}"
            )
        else:
            print("  [run] computing per-item aggregated features...")
            item_agg = (
                reviews_df.groupby("product_id")
                .agg(
                    num_topuser_reviews=("user_id", "count"),
                    num_unique_topusers=("user_id", "nunique"),
                    avg_rating_topusers=("rating", "mean"),
                    avg_helpful_votes_topusers=("helpful_votes", "mean"),
                    first_review_time=("unixReviewTime", "min"),
                    last_review_time=("unixReviewTime", "max"),
                )
                .reset_index()
            )

            def _get_meta_avg(asin):
                return item_meta.get(str(asin), {}).get("item_avg_rating")

            def _get_meta_cats(asin):
                return item_meta.get(str(asin), {}).get("item_categories")

            item_agg["item_avg_rating_meta"] = item_agg["product_id"].map(
                _get_meta_avg
            )
            item_agg["item_categories_meta"] = item_agg["product_id"].map(
                _get_meta_cats
            )

            item_agg.to_parquet(out_item_features, index=False)
            print(f"  [save] per-item features -> {out_item_features}")

        # ------------------------------------------------------------------
        # Step 4: EDA plots from rating/helpful hist
        # ------------------------------------------------------------------
        if not rating_png.exists() or not helpful_png.exists():
            print("  [run] creating EDA plots for top-user reviews...")
        if not rating_png.exists():
            save_rating_hist_plot(
                rating_hist,
                rating_png,
                title=f"Top users rating distribution ({category})",
            )
        else:
            print(f"  [skip] rating hist plot already exists: {rating_png}")

        if not helpful_png.exists():
            save_helpful_hist_plot(
                helpful_hist,
                helpful_png,
                title=f"Top users helpful votes ({category})",
            )
        else:
            print(f"  [skip] helpful votes plot already exists: {helpful_png}")

        # ------------------------------------------------------------------
        # Upload processed artifacts to Drive
        # ------------------------------------------------------------------
        upload_targets = [
            out_reviews_parquet,
            out_user_features,
            out_item_features,
            out_stats_json,
            rating_png,
            helpful_png,
        ]
        print("  [drive] uploading processed outputs to Drive...")
        for p in upload_targets:
            try:
                upload_to_drive(p)
            except Exception as e:
                print(f"  [drive] WARNING: failed to upload {p}: {e}")

        # ------------------------------------------------------------------
        # Cleanup processed outputs (optional)
        # ------------------------------------------------------------------
        if cleanup_processed != "none":
            for p in upload_targets:
                if cleanup_processed == "parquet" and p.suffix != ".parquet":
                    continue
                try:
                    if p.exists():
                        print(f"  [cleanup-processed] removing {p}")
                        p.unlink()
                except OSError as e:
                    print(f"  [cleanup-processed] WARNING: failed to remove {p}: {e}")

        # Cleanup raw gz if requested
        if cleanup_raw:
            for p in (review_gz, meta_gz):
                if p.exists():
                    print(f"  [cleanup-raw] removing {p}")
                    p.unlink()

        print(f"=== Done category: {category} ===")

    finally:
        # Always try to release lock
        if lock_path.exists():
            try:
                lock_path.unlink()
            except OSError:
                pass

        try:
            rel_lock = str(lock_path.relative_to(repo_root))
            delete_remote_by_rel_path(rel_lock)
        except Exception:
            # If Drive is down or the file is already gone, ignore
            pass


# --------------------------------------------------------------------
# Main
# --------------------------------------------------------------------
def main() -> None:
    args = parse_args()
    repo_root = get_repo_root()
    raw_dir = repo_root / "data" / "raw"
    processed_dir = repo_root / "data" / "processed"
    cleanup_raw = not (args.no_cleanup_raw or args.no_cleanup)
    cleanup_processed = args.cleanup_processed
    allow_download = not args.no_download

    # Always resync registry at the start so Drive state is fresh
    resync_registry()

    # Resolve top_users path
    if args.top_users:
        top_users_path = Path(args.top_users)
        if not top_users_path.is_absolute():
            top_users_path = repo_root / top_users_path
        if not top_users_path.exists():
            raise FileNotFoundError(
                f"Top users file not found: {top_users_path}"
            )
    else:
        # Use data registry (processed.top_users) and Google Drive if needed
        top_users_path = ensure_local("processed", "top_users")

    top_users_df = load_top_users(top_users_path)
    # Build a global set of user_ids once (string-typed) for fast membership checks
    top_users_set = set(top_users_df["user_id"].astype(str).tolist())

    # Determine categories
    if args.categories:
        categories = args.categories
        print(f"Using categories from CLI: {categories}")
    else:
        if args.categories_file:
            cat_file = Path(args.categories_file)
            if not cat_file.is_absolute():
                cat_file = repo_root / cat_file
            if not cat_file.exists():
                # Try drive registry lookup by relative path
                rel = str(cat_file.relative_to(repo_root))
                print(
                    f"[info] categories file not local; trying Drive for {rel}"
                )
                cat_file = ensure_local_path(rel)
        else:
            # Default: use registry entry raw.all_categories.txt
            cat_file = ensure_local("raw", "all_categories.txt")

        categories = read_all_categories_from_file(cat_file)
        print(f"No categories specified; using all from {cat_file}")
        print(f"{len(categories)} categories: {categories}")

    total_categories = len(categories)
    for idx, cat in enumerate(categories, start=1):
        print(f"\n>>> [{idx}/{total_categories}] Starting category: {cat}")
        try:
            process_category(
                cat,
                raw_dir=raw_dir,
                processed_dir=processed_dir,
                top_users_df=top_users_df,
                top_users_set=top_users_set,
                allow_download=allow_download,
                cleanup_raw=cleanup_raw,
                cleanup_processed=cleanup_processed,
                repo_root=repo_root,
                progress_interval=args.progress_interval,
            )
        except Exception as e:
            print(f"  [error] processing {cat}: {e}")


if __name__ == "__main__":
    main()

# usage examples:
# python common_scripts/03_user_features.py
# python common_scripts/03_user_features.py --categories Movies_and_TV Books
# python common_scripts/03_user_features.py --no-cleanup-raw
# python common_scripts/03_user_features.py --cleanup-processed none
# python common_scripts/03_user_features.py --cleanup-processed all

In [ ]:
#!/usr/bin/env python
"""
04_create_train_data.py

Build a *temporal* training set for:
    Given a user and their purchase/review history up to time t,
    predict the CATEGORY of their next purchase at time t+1.

Pipeline:

1. Ensure per-category processed outputs from script 3 exist:
   - data/processed/<cat>/top_user_reviews_<cat>.parquet
   - data/processed/<cat>/top_user_features_<cat>.parquet
   - (top_item_features_<cat>.parquet is not strictly needed here)

2. Load:
   - Combined per-review data across categories:
       user_id, product_id, unixReviewTime, rating, helpful_votes,
       helpful_votes_clipped, verified_purchase, item_avg_rating,
       item_categories, category (= top-level Amazon category)
   - Global user-level features (importance, entropy, etc.)

3. For each user:
   - Sort their reviews by unixReviewTime.
   - For each time step i where prefix length >= min_prefix
     and there exists a purchase at i+1:
       - Build features from *prefix* (reviews 0..i):
           * static user stats (importance, entropy, ...)
           * prefix_length, prefix_timespan
           * prefix_avg_rating, prefix_avg_helpful, prefix_avg_item_avg_rating
           * last rating/helpful/item_avg_rating/verified
           * last N categories (as integer indices)
           * prefix category counts & most frequent category
       - Label = category of purchase at i+1

4. Compute baselines:
   - Global majority category
   - Per-user majority category baseline
   - "Last category" baseline

5. Save:
   - data/global/sequence_training_samples.parquet
     (contains all features + target_category + target_category_idx)
"""

import argparse
import os
import shutil
from collections import Counter, defaultdict
from typing import Dict, List

import gc
import numpy as np
import pandas as pd
from tqdm import tqdm

import data_io


# ---------------------------------------------------------------------
# Data loading helpers
# ---------------------------------------------------------------------


class DataLoader:
    def ensure_categories_downloaded(self, categories: List[str]) -> List[str]:
        """
        Ensure that script-3 outputs exist locally (via Drive if needed).
        Returns the subset of categories for which all required files exist.
        Logs missing categories to data/processed/missing_categories.txt.
        """
        successful_categories = []
        missing_path = "data/processed/missing_categories.txt"
        os.makedirs(os.path.dirname(missing_path), exist_ok=True)

        for cat in categories:
            try:
                data_io.ensure_local_path(
                    f"data/processed/{cat}/top_user_reviews_{cat}.parquet"
                )
                data_io.ensure_local_path(
                    f"data/processed/{cat}/top_user_features_{cat}.parquet"
                )
                # Not strictly needed here, but we keep the check:
                data_io.ensure_local_path(
                    f"data/processed/{cat}/top_item_features_{cat}.parquet"
                )
                successful_categories.append(cat)
            except Exception:
                print(f"[warn] top-user outputs for category {cat} not fully available")
                with open(missing_path, "a", encoding="utf-8") as f:
                    f.write(f"{cat}\n")

        return successful_categories

    def load_categories_from_file(self, file_path: str) -> List[str]:
        """Load categories from a line-separated text file."""
        with open(file_path, "r", encoding="utf-8") as f:
            categories = [line.strip() for line in f if line.strip()]
        return categories

    def build_category_index(self, categories: List[str]) -> Dict[str, int]:
        """
        Build a mapping category_name -> integer index, with 0 reserved for 'Unknown'.
        """
        cat_to_idx = {"Unknown": 0}
        idx = 1
        for cat in categories:
            if cat not in cat_to_idx:
                cat_to_idx[cat] = idx
                idx += 1
        return cat_to_idx


# ---------------------------------------------------------------------
# Phase 0: shard user features by user_id to disk
# ---------------------------------------------------------------------


def shard_user_features(
    categories: List[str],
    tmp_user_dir: str,
    n_shards: int = 256,
) -> None:
    """
    For each category, load top_user_features_<cat>.parquet and write it into a
    partitioned Parquet dataset under tmp_user_dir, partitioned by 'user_shard'.

    Logic:
      - user_shard = hash_pandas_object(user_id) % n_shards
      - We do NOT concatenate across categories in memory.
      - Duplicates for the same user across categories will land in the same shard
        and will be deduplicated later per shard.
    """
    from pandas.util import hash_pandas_object
    import pyarrow as pa
    import pyarrow.parquet as pq

    os.makedirs(tmp_user_dir, exist_ok=True)

    for cat in categories:
        path = f"data/processed/{cat}/top_user_features_{cat}.parquet"
        if not os.path.exists(path):
            print(f"[warn] missing user features parquet for {cat}: {path}")
            continue

        print(f"[user-shard] loading user features for {cat} from {path}")
        df = pd.read_parquet(path)

        if "user_id" not in df.columns:
            print(f"[warn] {cat} user features missing user_id; skipping")
            continue

        df["user_id"] = df["user_id"].astype(str)

        # Compute shard id from user_id (vectorized)
        shard_ids = hash_pandas_object(df["user_id"], index=False).values % n_shards
        df["user_shard"] = shard_ids.astype("int32")

        print(f"[user-shard] writing {len(df):,} rows for {cat} into user shards...")
        table = pa.Table.from_pandas(df)

        pq.write_to_dataset(
            table,
            root_path=tmp_user_dir,
            partition_cols=["user_shard"],
        )

        del df, table
        gc.collect()

    print("[user-shard] done creating sharded user-features dataset.")


# ---------------------------------------------------------------------
# Phase 1: shard reviews by user_id to disk
# ---------------------------------------------------------------------


def shard_reviews_by_user(
    categories: List[str],
    category_index: Dict[str, int],
    tmp_dir: str,
    n_shards: int = 256,
) -> None:
    """
    For each category, load its top_user_reviews_<cat>.parquet and write it into a
    partitioned Parquet dataset under tmp_dir, partitioned by 'user_shard'.

    Logic:
      - For each review row, compute user_shard = hash(user_id) % n_shards
      - Write to Parquet dataset with partition_cols = ['user_shard'].

    Later we read one shard (partition) at a time to build sequences.
    """
    from pandas.util import hash_pandas_object
    import pyarrow as pa
    import pyarrow.parquet as pq

    os.makedirs(tmp_dir, exist_ok=True)

    for cat in categories:
        path = f"data/processed/{cat}/top_user_reviews_{cat}.parquet"
        if not os.path.exists(path):
            print(f"[warn] missing reviews parquet for {cat}: {path}")
            continue

        print(f"[review-shard] loading reviews for {cat} from {path}")
        df = pd.read_parquet(path)

        # We only need these columns; if any missing, create reasonable defaults.
        if "user_id" not in df.columns or "unixReviewTime" not in df.columns:
            print(f"[warn] {cat} missing user_id/unixReviewTime; skipping")
            continue

        df["user_id"] = df["user_id"].astype(str)
        df = df.dropna(subset=["unixReviewTime"])
        df["unixReviewTime"] = df["unixReviewTime"].astype("int64")

        if "rating" not in df.columns:
            df["rating"] = np.nan
        if "helpful_votes" not in df.columns:
            df["helpful_votes"] = 0
        if "item_avg_rating" not in df.columns:
            df["item_avg_rating"] = np.nan
        if "verified_purchase" not in df.columns:
            df["verified_purchase"] = False

        # Attach top-level category and its index
        df["category"] = cat
        df["category_idx"] = df["category"].map(
            lambda c: category_index.get(c, 0)
        )

        # Compute shard id from user_id (vectorized)
        shard_ids = hash_pandas_object(df["user_id"], index=False).values % n_shards
        df["user_shard"] = shard_ids.astype("int32")

        print(f"[review-shard] writing {len(df):,} rows for {cat} into review shards...")
        table = pa.Table.from_pandas(df)

        pq.write_to_dataset(
            table,
            root_path=tmp_dir,
            partition_cols=["user_shard"],
        )

        del df, table
        gc.collect()

    print("[review-shard] done creating sharded review dataset.")


def list_shard_dirs(tmp_dir: str) -> List[str]:
    """
    Return sorted list of shard partition directories under tmp_dir,
    e.g. tmp_dir/user_shard=0, tmp_dir/user_shard=1, ...
    """
    shard_dirs = []
    if not os.path.exists(tmp_dir):
        return shard_dirs

    for entry in os.scandir(tmp_dir):
        if entry.is_dir() and entry.name.startswith("user_shard="):
            shard_dirs.append(entry.path)

    shard_dirs.sort()
    return shard_dirs


# ---------------------------------------------------------------------
# Phase 2: build sequences per shard
# ---------------------------------------------------------------------


def build_sequence_dataset_for_shard(
    reviews_df: pd.DataFrame,
    user_features_df: pd.DataFrame,
    category_index: Dict[str, int],
    n_latest: int = 5,
    min_prefix: int = 3,
    disable_prefix_cat_counts: bool = False,
    sample_every_k_prefix: int = 1,
) -> pd.DataFrame:
    """
    Build sequence samples for a SINGLE shard:

      For each user in this shard and each time step i where prefix length >= min_prefix
      and there exists a purchase at i+1, create a training sample:

        - Features: prefix-based summary up to i (only past info)
        - Label: category at i+1

    Returns a DataFrame of samples for this shard.
    """

    if reviews_df.empty:
        return pd.DataFrame()

    reviews_df = reviews_df.copy()

    reviews_df["user_id"] = reviews_df["user_id"].astype(str)

    # Ensure required columns exist
    required_cols = ["user_id", "unixReviewTime", "category_idx"]
    for col in required_cols:
        if col not in reviews_df.columns:
            raise ValueError(f"shard reviews_df missing required column '{col}'")

    # Normalize and clean
    reviews_df = reviews_df.dropna(subset=["unixReviewTime"])
    reviews_df["unixReviewTime"] = reviews_df["unixReviewTime"].astype("int64")

    if "rating" not in reviews_df.columns:
        reviews_df["rating"] = np.nan
    if "helpful_votes" not in reviews_df.columns:
        reviews_df["helpful_votes"] = 0
    if "item_avg_rating" not in reviews_df.columns:
        reviews_df["item_avg_rating"] = np.nan
    if "verified_purchase" not in reviews_df.columns:
        reviews_df["verified_purchase"] = False
    if "category" not in reviews_df.columns:
        # Map category_idx back to name if needed
        idx_to_cat = {idx: name for name, idx in category_index.items()}
        reviews_df["category"] = reviews_df["category_idx"].map(
            lambda i: idx_to_cat.get(i, "Unknown")
        )

    # Static user features (per shard)
    uf = user_features_df.copy()
    if not uf.empty:
        uf["user_id"] = uf["user_id"].astype(str)
        uf = uf.drop_duplicates(subset=["user_id"], keep="first")
        uf = uf.set_index("user_id", drop=False)
        static_feature_cols = [c for c in uf.columns if c != "user_id"]
    else:
        uf = pd.DataFrame(columns=["user_id"]).set_index("user_id", drop=False)
        static_feature_cols = []

    # Group by user within this shard
    #reviews_df = reviews_df.sort_values(["user_id", "unixReviewTime"])
    # Group by user within this shard
    # NOTE: global sort is unnecessary (we sort per user below)
    # and was causing a Categorical bug on some shards.
    grouped = reviews_df.groupby("user_id", sort=False)

    num_cats = max(category_index.values()) if category_index else 0
    samples = []

    # Pull locals for a tiny speed boost
    np_isnan = np.isnan
    pd_isna = pd.isna
    category_items = list(category_index.items())

    for user_id, user_group in grouped:
        user_group = user_group.sort_values("unixReviewTime")
        n = len(user_group)
        if n <= min_prefix:
            continue

        times = user_group["unixReviewTime"].to_numpy()
        cat_idxs = user_group["category_idx"].to_numpy()
        ratings = user_group["rating"].to_numpy()
        helpful = user_group["helpful_votes"].to_numpy()
        item_avg = user_group["item_avg_rating"].to_numpy()
        verified = user_group["verified_purchase"].to_numpy()
        target_cats = user_group["category"].astype(str).to_numpy()

        if n == 0:
            continue

        # Prefix accumulators
        prefix_len = 0
        sum_rating = 0.0
        rating_count = 0
        sum_helpful = 0.0
        sum_item_avg = 0.0
        item_avg_count = 0
        cat_counts = np.zeros(num_cats + 1, dtype=np.int64)
        first_time = None

        # Static user features (if present)
        if user_id in uf.index:
            urow = uf.loc[user_id]
        else:
            urow = None

        for i in range(n):
            t = int(times[i])
            cidx = int(cat_idxs[i])
            r = ratings[i]
            h = helpful[i]
            ia = item_avg[i]
            v = bool(verified[i])

            prefix_len += 1
            if first_time is None:
                first_time = t
            cat_counts[cidx] += 1

            if not pd_isna(r):
                sum_rating += float(r)
                rating_count += 1
            if not pd_isna(h):
                sum_helpful += float(h)
            if not pd_isna(ia):
                sum_item_avg += float(ia)
                item_avg_count += 1

            # Need at least min_prefix purchases so far, and a future one at i+1
            if prefix_len < min_prefix or i >= n - 1:
                continue

            # Downsample prefixes: only keep every k-th prefix if k > 1
            if sample_every_k_prefix > 1:
                # Example: first eligible prefix (prefix_len == min_prefix) is kept,
                # then every k-th afterwards.
                if (prefix_len - min_prefix) % sample_every_k_prefix != 0:
                    continue

            feat = {}

            # 1) ID
            feat["user_id"] = user_id

            # 2) Static user features
            if urow is not None:
                for col in static_feature_cols:
                    feat[col] = urow.get(col, np.nan)
            else:
                for col in static_feature_cols:
                    feat[col] = np.nan

            # 3) Prefix-level summary
            feat["prefix_length"] = prefix_len
            feat["prefix_timespan"] = (
                t - first_time if first_time is not None else 0
            )
            feat["prefix_avg_rating"] = (
                sum_rating / rating_count if rating_count > 0 else np.nan
            )
            feat["prefix_avg_helpful"] = (
                sum_helpful / prefix_len if prefix_len > 0 else 0.0
            )
            feat["prefix_avg_item_avg_rating"] = (
                sum_item_avg / item_avg_count if item_avg_count > 0 else np.nan
            )

            # 4) Last purchase features at time i
            feat["last_category_idx"] = cidx
            feat["last_rating"] = float(r) if not pd_isna(r) else np.nan
            feat["last_helpful_votes"] = float(h) if not pd_isna(h) else 0.0
            feat["last_item_avg_rating"] = (
                float(ia) if not pd_isna(ia) else np.nan
            )
            feat["last_verified"] = int(v)

            # 5) Last n_latest categories (backwards from i)
            for k in range(n_latest):
                j = i - k
                if j >= 0:
                    feat[f"last_{k+1}_category_idx"] = int(cat_idxs[j])
                else:
                    feat[f"last_{k+1}_category_idx"] = 0

            # 6) Prefix category counts
            if not disable_prefix_cat_counts:
                for cat_name, idx in category_items:
                    if cat_name == "Unknown":
                        feat["prefix_cat_count_Unknown"] = int(cat_counts[idx])
                    else:
                        feat[f"prefix_cat_count_{cat_name}"] = int(cat_counts[idx])

            # 7) Most frequent category in prefix
            feat["prefix_most_freq_category_idx"] = int(cat_counts.argmax())

            # 8) Target: category at i+1
            next_idx = int(cat_idxs[i + 1])
            next_cat_name = target_cats[i + 1]

            feat["target_category_idx"] = next_idx
            feat["target_category"] = next_cat_name

            samples.append(feat)

    if not samples:
        return pd.DataFrame()

    return pd.DataFrame(samples)


# ---------------------------------------------------------------------
# Baseline stats (streaming)
# ---------------------------------------------------------------------


class BaselineStats:
    """
    Streaming baselines:

      - global majority category
      - per-user majority category
      - last-category baseline
    """

    def __init__(self):
        self.global_counts = Counter()                  # category_name -> count
        self.user_label_counts = defaultdict(Counter)   # user_id -> Counter(label_idx -> count)
        self.total_samples = 0
        self.last_equal_correct = 0

    def update_from_shard(self, shard_df: pd.DataFrame) -> None:
        if shard_df.empty:
            return

        self.total_samples += len(shard_df)

        # Global category counts (by name)
        self.global_counts.update(shard_df["target_category"].tolist())

        # Per-user label counts (by idx)
        for user_id, grp in shard_df.groupby("user_id"):
            counts = grp["target_category_idx"].value_counts()
            for label_idx, cnt in counts.items():
                self.user_label_counts[user_id][int(label_idx)] += int(cnt)

        # Last-category baseline (predict last_category_idx)
        self.last_equal_correct += int(
            (shard_df["last_category_idx"] == shard_df["target_category_idx"]).sum()
        )

    def log(self) -> None:
        if self.total_samples == 0:
            print("[baseline] no samples; cannot compute baselines.")
            return

        # Global majority
        majority_cat, majority_cnt = max(self.global_counts.items(), key=lambda kv: kv[1])
        majority_acc = majority_cnt / self.total_samples

        # Per-user majority
        correct_user_majority = 0
        for counts in self.user_label_counts.values():
            if counts:
                correct_user_majority += max(counts.values())
        user_majority_acc = correct_user_majority / self.total_samples

        # Last-category baseline
        last_cat_acc = self.last_equal_correct / self.total_samples

        print("\n[baseline] Global majority category:")
        print(f"  majority_category = {majority_cat}")
        print(f"  accuracy          = {majority_acc:.4f}")

        print("\n[baseline] Per-user majority category:")
        print(f"  accuracy          = {user_majority_acc:.4f}")

        print("\n[baseline] Last-category baseline:")
        print(f"  accuracy          = {last_cat_acc:.4f}")

        print("\n[baseline] summary:")
        print(f"  Global majority:   {majority_acc:.4f}")
        print(f"  Per-user majority: {user_majority_acc:.4f}")
        print(f"  Last-category:     {last_cat_acc:.4f}\n")


# ---------------------------------------------------------------------
# CLI / main
# ---------------------------------------------------------------------


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Build temporal training data (streaming) to predict next purchase category."
    )
    parser.add_argument(
        "--categories-file",
        type=str,
        default="data/raw/all_categories.txt",
        help="Path to all_categories.txt (one category name per line).",
    )
    parser.add_argument(
        "--categories",
        nargs="*",
        help="Optional explicit list of categories (overrides --categories-file).",
    )
    parser.add_argument(
        "--n-latest",
        type=int,
        default=5,
        help="Number of last purchases to encode as features.",
    )
    parser.add_argument(
        "--min-prefix",
        type=int,
        default=3,
        help="Minimum prefix length to create a sample.",
    )
    parser.add_argument(
        "--n-shards",
        type=int,
        default=256,
        help="Number of user shards for streaming processing.",
    )
    parser.add_argument(
        "--tmp-dir",
        type=str,
        default="data/tmp/review_shards",
        help="Temporary directory to store sharded review data.",
    )
    parser.add_argument(
        "--tmp-user-dir",
        type=str,
        default="data/tmp/user_shards",
        help="Temporary directory to store sharded user feature data.",
    )
    parser.add_argument(
        "--output-path",
        type=str,
        default="data/global/sequence_training_samples.parquet",
        help="Where to write the final sequence dataset parquet.",
    )
    parser.add_argument(
        "--skip-user-sharding",
        action="store_true",
        help="Skip Phase 0 and reuse existing user_shards if present.",
    )
    parser.add_argument(
        "--skip-review-sharding",
        action="store_true",
        help="Skip Phase 1 and reuse existing review_shards if present.",
    )
    parser.add_argument(
        "--disable-prefix-cat-counts",
        action="store_true",
        help="If set, do NOT materialize per-category prefix counts (prefix_cat_count_*)."
    )
    parser.add_argument(
        "--sample-every-k-prefix",
        type=int,
        default=1,
        help="Only create a sequence sample for every k-th prefix (per user). "
             "k=1 means use every possible prefix; k=5 means keep ~1/5 of them."
    )
    parser.add_argument(
        "--disable-baselines",
        action="store_true",
        help="If set, skip computing streaming baselines on the full dataset."
    )
    parser.add_argument(
        "--per-shard-output-dir",
        type=str,
        default="data/global/sequence_samples_by_shard",
        help="Directory to write per-shard parquet outputs for Phase 2.",
    )
    parser.add_argument(
        "--resume-phase2",
        action="store_true",
        help="If set, Phase 2 will skip shards whose per-shard parquet already exists.",
    )
    return parser


def main() -> None:
    parser = build_arg_parser()
    args = parser.parse_args()

    data_io.resync_registry()

    loader = DataLoader()

    # Determine categories
    if args.categories:
        categories = args.categories
        print(f"[info] using categories from CLI: {categories}")
    else:
        categories = loader.load_categories_from_file(args.categories_file)
        print(f"[info] loaded {len(categories)} categories from {args.categories_file}")

    # Ensure script-3 outputs exist locally / via Drive
    categories_ok = loader.ensure_categories_downloaded(categories)
    print(f"[info] categories with complete script-3 outputs: {len(categories_ok)}")
    if not categories_ok:
        raise RuntimeError("No categories have the required script-3 outputs.")

    # Build category index (for All_Beauty, Amazon_Fashion, etc.)
    category_index = loader.build_category_index(categories_ok)
    print(f"[info] category -> idx mapping: {category_index}")

    # Phase 0: shard user features
    if args.skip_user_sharding:
        if not os.path.exists(args.tmp_user_dir):
            raise RuntimeError(
                f"--skip-user-sharding was set but tmp_user_dir={args.tmp_user_dir} does not exist."
            )
        print(f"\n=== Phase 0: Skipping user sharding (reusing {args.tmp_user_dir}) ===")
    else:
        if os.path.exists(args.tmp_user_dir):
            print(f"[info] removing existing tmp_user_dir={args.tmp_user_dir} to avoid stale shards")
            shutil.rmtree(args.tmp_user_dir)

        print("\n=== Phase 0: Sharding user features by user_id ===")
        shard_user_features(
            categories=categories_ok,
            tmp_user_dir=args.tmp_user_dir,
            n_shards=args.n_shards,
        )

    # Phase 1: shard all reviews by user_id
    if args.skip_review_sharding:
        if not os.path.exists(args.tmp_dir):
            raise RuntimeError(
                f"--skip-review-sharding was set but tmp_dir={args.tmp_dir} does not exist."
            )
        print(f"\n=== Phase 1: Skipping review sharding (reusing {args.tmp_dir}) ===")
    else:
        if os.path.exists(args.tmp_dir):
            print(f"[info] removing existing tmp_dir={args.tmp_dir} to avoid stale shards")
            shutil.rmtree(args.tmp_dir)

        print("\n=== Phase 1: Sharding reviews by user_id ===")
        shard_reviews_by_user(
            categories=categories_ok,
            category_index=category_index,
            tmp_dir=args.tmp_dir,
            n_shards=args.n_shards,
        )

    shard_dirs = list_shard_dirs(args.tmp_dir)
    num_shards = len(shard_dirs)
    print(f"[info] found {num_shards} review shard partitions in {args.tmp_dir}")

    if not shard_dirs:
        raise RuntimeError("No shard directories found; sharding may have failed.")

    # Phase 2: For each shard, build sequence samples and stream to Parquet
    import pyarrow as pa
    import pyarrow.parquet as pq

    # Make sure per-shard output directory exists
    os.makedirs(args.per_shard_output_dir, exist_ok=True)
    baseline_stats = BaselineStats()

    print("\n=== Phase 2: Building sequence samples per shard ===")
    shard_idx = 0
    for shard_dir in tqdm(shard_dirs, desc="Shards"):
        shard_idx += 1
        shard_name = os.path.basename(shard_dir)  # e.g., "user_shard=123"
        print(f"\n[phase2] === Shard {shard_idx}/{num_shards}: {shard_name} ===")

        # Where we store this shard's sequence samples
        shard_output_path = os.path.join(
            args.per_shard_output_dir,
            f"sequence_{shard_name}.parquet",
        )

        # -----------------------------------------------------------------
        # Upgrade-aware resume logic:
        #   - If resume_phase2 and file exists, we inspect whether it already
        #     contains prefix_cat_count_* columns.
        #   - If prefix counts are required (disable_prefix_cat_counts=False)
        #     but missing, we rebuild this shard.
        #   - Otherwise, we reuse the existing shard and optionally update
        #     baselines from it.
        # -----------------------------------------------------------------
        if os.path.exists(shard_output_path) and args.resume_phase2:
            print(f"[phase2] shard {shard_name}: found existing {shard_output_path}, inspecting schema...")

            # Cheaply inspect schema via pyarrow without loading full data
            pf = pq.ParquetFile(shard_output_path)
            col_names = pf.schema.names
            has_prefix_counts = any(
                name.startswith("prefix_cat_count_") for name in col_names
            )

            # Case 1: User *doesn't* want prefix counts this run
            if args.disable_prefix_cat_counts:
                print(f"[phase2] shard {shard_name}: reusing existing shard "
                      f"(prefix_cat_counts disabled for this run).")

                if not args.disable_baselines:
                    print(f"[phase2] shard {shard_name}: loading existing shard for baselines...")
                    existing_df = pf.read().to_pandas()
                    baseline_stats.update_from_shard(existing_df)
                    print(f"[phase2] shard {shard_name}: reused {len(existing_df):,} samples for baselines.")
                    del existing_df
                    gc.collect()

                # Skip heavy work
                continue

            # Case 2: User *does* want prefix counts and they already exist
            if has_prefix_counts:
                print(f"[phase2] shard {shard_name}: already has prefix_cat_count_* features; reusing.")

                if not args.disable_baselines:
                    print(f"[phase2] shard {shard_name}: loading existing shard for baselines...")
                    existing_df = pf.read().to_pandas()
                    baseline_stats.update_from_shard(existing_df)
                    print(f"[phase2] shard {shard_name}: reused {len(existing_df):,} samples for baselines.")
                    del existing_df
                    gc.collect()

                # Skip heavy work
                continue

            # Case 3: User wants prefix counts but they are missing -> rebuild
            print(f"[phase2] shard {shard_name}: existing file lacks prefix_cat_count_*; "
                  f"rebuilding this shard with new features (will overwrite).")

        # If we got here, either:
        #   - shard_output_path doesn't exist, or
        #   - resume_phase2=False, or
        #   - we explicitly decided to rebuild because we are upgrading
        #     to include prefix_cat_count_* features.

        # Load review shard
        rfiles = [
            os.path.join(shard_dir, f)
            for f in os.listdir(shard_dir)
            if f.endswith(".parquet")
        ]
        if not rfiles:
            print(f"[phase2] shard {shard_name}: no review parquet files, skipping.")
            continue

        print(f"[phase2] shard {shard_name}: loading {len(rfiles)} review files...")
        shard_dfs = [pd.read_parquet(p) for p in rfiles]
        shard_reviews = pd.concat(shard_dfs, ignore_index=True)
        del shard_dfs
        gc.collect()

        num_rows = len(shard_reviews)
        num_users = shard_reviews["user_id"].nunique() if "user_id" in shard_reviews.columns else 0
        print(f"[phase2] shard {shard_name}: {num_rows:,} rows, {num_users:,} users")

        # Load matching user-feature shard (if exists)
        user_shard_dir = os.path.join(args.tmp_user_dir, shard_name)
        if os.path.exists(user_shard_dir):
            ufiles = [
                os.path.join(user_shard_dir, f)
                for f in os.listdir(user_shard_dir)
                if f.endswith(".parquet")
            ]
            if ufiles:
                print(f"[phase2] shard {shard_name}: loading {len(ufiles)} user-feature files...")
                udfs = [pd.read_parquet(p) for p in ufiles]
                shard_user_features_df = pd.concat(udfs, ignore_index=True)
                del udfs
                gc.collect()
            else:
                print(f"[phase2] shard {shard_name}: no user-feature files (using empty).")
                shard_user_features_df = pd.DataFrame(columns=["user_id"])
        else:
            print(f"[phase2] shard {shard_name}: user-feature shard dir not found (using empty).")
            shard_user_features_df = pd.DataFrame(columns=["user_id"])

        print(f"[phase2] shard {shard_name}: building sequence dataset...")
        seq_df_shard = build_sequence_dataset_for_shard(
            reviews_df=shard_reviews,
            user_features_df=shard_user_features_df,
            category_index=category_index,
            n_latest=args.n_latest,
            min_prefix=args.min_prefix,
            disable_prefix_cat_counts=args.disable_prefix_cat_counts,
            sample_every_k_prefix=args.sample_every_k_prefix,
        )
        del shard_reviews, shard_user_features_df
        gc.collect()

        if seq_df_shard.empty:
            print(f"[phase2] shard {shard_name}: no valid sequence samples, skipping.")
            continue

        shard_samples = len(seq_df_shard)
        print(f"[phase2] shard {shard_name}: created {shard_samples:,} sequence samples")

        # Update baselines (optional)
        if not args.disable_baselines:
            baseline_stats.update_from_shard(seq_df_shard)
            print(f"[phase2] cumulative samples so far (for baselines): {baseline_stats.total_samples:,}")
        else:
            print(f"[phase2] cumulative samples so far (no baselines): "
                  f"{shard_samples:,} this shard")

        # Write / overwrite per-shard output
        table = pa.Table.from_pandas(seq_df_shard)
        pq.write_table(table, shard_output_path)
        print(f"[phase2] shard {shard_name}: wrote per-shard output -> {shard_output_path}")

        del seq_df_shard, table
        gc.collect()

    # -----------------------------------------------------------------
    # Combine per-shard outputs into a single global Parquet file
    # -----------------------------------------------------------------
    import pyarrow.parquet as pq
    import pyarrow as pa

    shard_files = [
        os.path.join(args.per_shard_output_dir, f)
        for f in os.listdir(args.per_shard_output_dir)
        if f.endswith(".parquet")
    ]
    shard_files.sort()

    if not shard_files:
        print("[save] No per-shard outputs found; global output will not be written.")
    else:
        print(f"\n[save] combining {len(shard_files)} shard files into {args.output_path} ...")
        os.makedirs(os.path.dirname(args.output_path), exist_ok=True)

        # If a previous global file exists (e.g., from a failed run), remove it
        if os.path.exists(args.output_path):
            print(f"[save] removing existing global output at {args.output_path} to avoid duplicates.")
            os.remove(args.output_path)

        writer = None
        total_rows = 0

        for fpath in shard_files:
            df = pd.read_parquet(fpath)
            total_rows += len(df)
            table = pa.Table.from_pandas(df)
            if writer is None:
                writer = pq.ParquetWriter(args.output_path, table.schema)
            writer.write_table(table)
            del df, table
            gc.collect()

        if writer is not None:
            writer.close()
            print(f"[save] sequence training samples -> {args.output_path} "
                  f"(total rows: {total_rows:,})")
            try:
                abs_path = os.path.abspath(args.output_path)
                data_io.upload_to_drive(abs_path)
                print(f"[drive] uploaded sequence dataset to Drive: {abs_path}")
            except Exception as e:
                print(f"[drive] WARNING: failed to upload sequence dataset: {e}")

    # Log baselines (global, over all shards)
    if not args.disable_baselines:
        baseline_stats.log()
    else:
        print("[baseline] skipped baseline computation (disable_baselines=True).")


if __name__ == "__main__":
    main()

# python common_scripts/04_create_train_data.py --resume-phase2
"""
python common_scripts/04_create_train_data.py \
--skip-user-sharding \
--skip-review-sharding \
--resume-phase2

python common_scripts/04_create_train_data.py --skip-user-sharding --skip-review-sharding --sample-every-k-prefix 10 --resume-phase2

python common_scripts/04_create_train_data.py --skip-user-sharding --skip-review-sharding --disable-prefix-cat-counts --sample-every-k-prefix 5 --resume-phase2
"""

In [ ]:
#!/usr/bin/env python
"""
train_logreg_next_category.py

Train a multinomial logistic regression model to predict
target_category_idx from the sharded sequence data produced
by 04_create_train_data.py.

Instead of loading the huge global file
  data/global/sequence_training_samples.parquet
(which can blow up RAM), we work from the per-shard files:

  data/global/sequence_samples_by_shard/sequence_user_shard=*.parquet

Design:

- Shard-level split (equivalent to user-level split, since shards are
  defined by hash(user_id)):
    * 80% of shards -> train
    * 10% of shards -> val
    * 10% of shards -> test
- We cap the *total rows* for each split to stay within memory.
- Modeling approach:
    * Numeric features: SimpleImputer + StandardScaler
    * *_category_idx features: SimpleImputer + OneHotEncoder
    * Classifier: multinomial LogisticRegression (lbfgs)
- Baselines on the validation set:
    * Global majority class
    * last_category_idx
    * prefix_most_freq_category_idx
"""

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    top_k_accuracy_score,
    classification_report,
)

# ---------------------------------------------------------------------
# Wire up common_scripts.data_io regardless of where we run from
# ---------------------------------------------------------------------

THIS_DIR = Path(__file__).resolve().parent
REPO_ROOT = THIS_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common_scripts import data_io  # type: ignore

# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

SHARD_DIR = "data/global/sequence_samples_by_shard"
GLOBAL_OUT_PATH = "data/global/sequence_training_samples.parquet"  # uploaded by 04_create_train_data.py

RANDOM_SEED = 42

# Target total rows per split (approx upper bound)
MAX_TRAIN_ROWS = 1_000_000
MAX_VAL_ROWS   =   200_000
MAX_TEST_ROWS  =   200_000

# Fractions of shards for each split
TRAIN_FRAC = 0.8
VAL_FRAC   = 0.1  # rest -> test

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------


def list_shard_files(shard_dir: str):
    if not os.path.exists(shard_dir):
        raise FileNotFoundError(
            f"Shard directory '{shard_dir}' not found. "
            f"Make sure 04_create_train_data.py was run with per-shard outputs enabled."
        )

    shard_files = [
        os.path.join(shard_dir, f)
        for f in os.listdir(shard_dir)
        if f.endswith(".parquet") and f.startswith("sequence_user_shard=")
    ]
    shard_files.sort()
    if not shard_files:
        raise RuntimeError(
            f"No shard files found under {shard_dir}. "
            f"Expected files like 'sequence_user_shard=0.parquet'."
        )
    return shard_files


def load_split_from_shards(files, max_rows: int, name: str):
    """
    Load up to max_rows from the given list of shard Parquet files.
    If max_rows is None, load all rows.
    """
    print(f"\n[load:{name}] Loading split from {len(files)} shard files "
          f"(max_rows={max_rows if max_rows is not None else 'ALL'}) ...")

    dfs = []
    total = 0

    for fpath in files:
        df_shard = pd.read_parquet(fpath)

        if max_rows is not None:
            remaining = max_rows - total
            if remaining <= 0:
                break
            if len(df_shard) > remaining:
                df_shard = df_shard.sample(n=remaining, random_state=RANDOM_SEED)

        dfs.append(df_shard)
        total += len(df_shard)
        print(f"[load:{name}]   +{len(df_shard):,} rows from {os.path.basename(fpath)} "
              f"(cumulative: {total:,})")

        if max_rows is not None and total >= max_rows:
            break

    if not dfs:
        raise RuntimeError(f"[load:{name}] No rows loaded for split '{name}'.")

    df_out = pd.concat(dfs, ignore_index=True)
    print(f"[load:{name}] Final {name} size: {len(df_out):,} rows")
    return df_out


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------


def main():
    np.random.seed(RANDOM_SEED)

    # 0. Resync data registry and ensure global sequence file is available (if registered)
    #    This makes it easy to run on a fresh machine: the global parquet will be
    #    downloaded from Drive if the registry knows about it.
    print("[data_io] Resyncing registry and trying to ensure global sequence dataset is local...")
    try:
        data_io.resync_registry()
        data_io.ensure_local_path(GLOBAL_OUT_PATH)
        print(f"[data_io] ensure_local_path OK for {GLOBAL_OUT_PATH}")
    except Exception as e:
        print(f"[data_io] WARNING: could not ensure {GLOBAL_OUT_PATH}: {e}")

    # 1. Discover shard files
    print(f"[shards] Listing shard files under {SHARD_DIR} ...")
    shard_files = list_shard_files(SHARD_DIR)
    n_shards = len(shard_files)
    print(f"[shards] Found {n_shards} shard files.")

    # Shuffle shards for randomness
    rng = np.random.RandomState(RANDOM_SEED)
    shard_files = np.array(shard_files)
    rng.shuffle(shard_files)

    # Split shards into train/val/test
    n_train = int(TRAIN_FRAC * n_shards)
    n_val = int(VAL_FRAC * n_shards)
    n_test = n_shards - n_train - n_val

    train_files = shard_files[:n_train]
    val_files   = shard_files[n_train:n_train + n_val]
    test_files  = shard_files[n_train + n_val:]

    print(f"[shards] Split into:")
    print(f"  train shards: {len(train_files)}")
    print(f"  val shards:   {len(val_files)}")
    print(f"  test shards:  {len(test_files)}")

    # 2. Load splits (respecting max rows)
    df_train = load_split_from_shards(train_files, MAX_TRAIN_ROWS, "train")
    df_val   = load_split_from_shards(val_files,   MAX_VAL_ROWS,   "val")
    df_test  = load_split_from_shards(test_files,  MAX_TEST_ROWS,  "test")

    # Basic sanity checks
    for name, df_part in [("train", df_train), ("val", df_val), ("test", df_test)]:
        for col in ["user_id", "target_category_idx", "target_category"]:
            if col not in df_part.columns:
                raise ValueError(f"[{name}] Missing required column '{col}'")
        df_part["target_category_idx"] = df_part["target_category_idx"].astype(int)

    # 3. Baselines on VAL set
    print("\n[baseline] Computing baselines on VAL split ...")

    y_val_true = df_val["target_category_idx"].to_numpy()

    # Global majority (from TRAIN)
    majority_class = df_train["target_category_idx"].value_counts().idxmax()
    y_val_majority = np.full_like(y_val_true, fill_value=majority_class)
    acc_majority = accuracy_score(y_val_true, y_val_majority)

    print(f"[baseline] Global majority class idx = {majority_class}")
    print(f"[baseline] Val accuracy (global majority) = {acc_majority:.4f}")

    # Last-category baseline
    if "last_category_idx" in df_val.columns:
        y_val_last = df_val["last_category_idx"].astype(int).to_numpy()
        acc_last = accuracy_score(y_val_true, y_val_last)
        print(f"[baseline] Val accuracy (last_category_idx) = {acc_last:.4f}")
    else:
        print("[baseline] last_category_idx not found; skipping last-category baseline.")

    # Prefix-most-frequent-category baseline
    if "prefix_most_freq_category_idx" in df_val.columns:
        y_val_most = df_val["prefix_most_freq_category_idx"].astype(int).to_numpy()
        acc_most = accuracy_score(y_val_true, y_val_most)
        print(f"[baseline] Val accuracy (prefix_most_freq_category_idx) = {acc_most:.4f}")
    else:
        print("[baseline] prefix_most_freq_category_idx not found; skipping that baseline.")

    # 4. Feature selection
    print("\n[fe] Preparing feature matrices ...")

    label_col = "target_category_idx"
    drop_cols = ["user_id", "target_category"]

    all_cols = df_train.columns.tolist()
    feature_cols = [c for c in all_cols if c not in drop_cols + [label_col]]

    X_train = df_train[feature_cols].copy()
    y_train = df_train[label_col].copy()

    X_val = df_val[feature_cols].copy()
    y_val = df_val[label_col].copy()

    X_test = df_test[feature_cols].copy()
    y_test = df_test[label_col].copy()

    # Categorical index-like features
    cat_feature_cols = [c for c in feature_cols if "category_idx" in c]
    numeric_feature_cols = [c for c in feature_cols if c not in cat_feature_cols]

    print(f"[fe] Total feature columns: {len(feature_cols)}")
    print(f"[fe] Numeric features    ({len(numeric_feature_cols)}): {numeric_feature_cols}")
    print(f"[fe] Categorical idx features ({len(cat_feature_cols)}): {cat_feature_cols}")

    # 5. Preprocessing + model
    print("\n[model] Building preprocessing pipeline + multinomial logistic regression ...")

    # ---- NEW: Imputation to handle NaNs before scaler / one-hot ----
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_feature_cols),
            ("cat", categorical_transformer, cat_feature_cols),
        ]
    )

    clf = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=200,
        n_jobs=-1,
        verbose=1,
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("clf", clf),
        ]
    )

    # 6. Train
    print("\n[train] Fitting model on TRAIN split ...")
    model.fit(X_train, y_train)
    print("[train] Done.")

    # 7. Evaluation helpers
    def evaluate_split(name, X, y_true):
        print(f"\n[eval:{name}] Evaluating on {name} set (n={len(y_true):,}) ...")
        y_pred = model.predict(X)
        acc = accuracy_score(y_true, y_pred)
        print(f"[eval:{name}] Accuracy = {acc:.4f}")

        # Top-3 accuracy (if predict_proba is available)
        try:
            y_proba = model.predict_proba(X)
            top3 = top_k_accuracy_score(y_true, y_proba, k=3)
            print(f"[eval:{name}] Top-3 accuracy = {top3:.4f}")
        except Exception as e:
            print(f"[eval:{name}] Could not compute top-3 accuracy: {e}")

        return y_pred, acc

    # 8. Evaluate on splits
    y_train_pred, train_acc = evaluate_split("train", X_train, y_train)
    y_val_pred,   val_acc   = evaluate_split("val",   X_val,   y_val)
    y_test_pred,  test_acc  = evaluate_split("test",  X_test,  y_test)

    print("\n[summary] Accuracies:")
    print(f"  Train: {train_acc:.4f}")
    print(f"  Val:   {val_acc:.4f}")
    print(f"  Test:  {test_acc:.4f}")

    # Detailed classification report on test
    print("\n[report:test] Classification report on TEST split:")
    print(classification_report(y_test, y_test_pred, digits=4))

    print("\nAll done.")


if __name__ == "__main__":
    main()

# python -m modeling.train_logreg_next_category